# Geospatial: how Europe grew online

Eurostat data, 2002–2025. Five chapters, each with an interactive plotly chart:
hover, play the animations, click the buttons. The last cell exports everything
below into a single self-contained HTML file (works in any browser, no login or
software needed) and downloads it.

## About the data

Four Eurostat datasets, covering 38 European countries, 2002–2025:

- **Household internet access** (`isoc_ci_in_h`): % of households with
  internet access, meaning a private home connection only, not access via
  work, school, hotels or cafes. 38 countries, 2002–2025.
- **Individual internet use** (`isoc_ci_ifp_iu`): % of individuals who used
  the internet in the last 3 months, anywhere, not just at home. 38
  countries, 2002–2025; also available split by age group (16-24 / 25-54 /
  55-74), but only from 2021 onward.
- **Digital activities** (`isoc_sk_dskl_i21`): % of individuals reaching "at
  least basic" across five activity areas, by age group. 37 countries,
  only 2021 / 2023 / 2025 (a newer, biennial indicator, not a continuous
  yearly series like the other two).
- **Digital activities, "above basic" only** (`isoc_sk_dskl_i21`,
  `indic_is=I_DSK2_AB`): the stricter "above basic" level on its own (all
  five DigComp areas at once, not just "basic or above" in each), by age
  group. Same 37 countries plus the EU27 and euro-area aggregates, same
  three years as above. Used by the closer look near the end of chapter 4.

- **Country boundaries**:
  - **GISCO**: all countries except Kosovo\*, Iceland and the United Kingdom.
  - **OpenStreetMap**:
    - **Kosovo\***: added, because Eurostat's own boundary data leaves
      Kosovo as part of Serbia (five EU member states don't recognize its
      independence). The asterisk follows the same standard EU/UN
      convention wherever Kosovo is named elsewhere in this project.
    - **Iceland and the United Kingdom**: replaced, because their Eurostat
      coastlines didn't align cleanly with the map underneath, leaving
      visible gaps.

### Why "digital activities," not "digital skills"

Eurostat's own name for the third dataset above is the **Digital Skills
Indicator 2.0**, but it is not a skills test. Respondents aged 16–74 are
simply asked yes/no whether they performed specific activities in the past
three months (twelve months for online purchases); Eurostat's own
methodology treats having done the activity as a proxy for having the
underlying skill, one that only runs in one direction: 'doing something'
implies 'being able to do it', but 'not doing it' implies nothing, someone
may simply have no current occasion to. A retiree who used spreadsheets for
years at work but hasn't needed to since scores the same as someone who
never learned how. Because the actual measurement is activity-based, this
project uses "digital activities" everywhere in its own writing, except when
naming Eurostat's dataset or indicator directly.

Activities are grouped into five DigComp 2.0 areas. What counts within each,
and how many of an area's own activities someone needs for a "basic" or
"above basic" score in that specific area:

| Area | Activities counted | Basic | Above basic |
|---|---|---|---|
| Information & data literacy | finding information on goods/services; seeking health information; reading online news; fact-checking online information and its sources | 1 activity | more than 1 |
| Communication & collaboration | email; internet phone/video calls; instant messaging; social networks; expressing opinions on civic/political issues online; online consultations or voting | 1 activity | more than 1 |
| Digital content creation | word processor; spreadsheet; editing photo/video/audio; copying or moving files between folders, devices or cloud; creating documents combining text, images, tables, charts; advanced spreadsheet features (formulas, macros); writing code | 1-2 activities | 3 or more |
| Safety | checking a site was secure before giving personal data; reading privacy statements; restricting access to geographical location; limiting access to a social-media profile; refusing use of personal data for advertising; changing browser cookie settings | 1-2 activities | 3 or more |
| Problem solving | installing software/apps; changing device or app settings; online purchases; selling online; using online learning resources; internet banking; job search or applications | 1-2 activities | 3 or more |

Reaching "basic" or "above basic" in one area, both measured over the same
three-month window as the rest of this indicator, is a per-area score.
Eurostat then combines all five per-area scores into one *overall* level,
across all five areas at once, reusing the same "basic" and "above basic"
labels one level up:

| Overall level | Eurostat's own definition |
|---|---|
| No skills | "no skills" in 4 or all 5 areas, despite having used the internet at least once in the last 3 months. |
| Low | "basic" or "above basic" level in 4 areas and "no skills" in 1 area (4 out of 5). |
| Basic | all 5 areas at least "basic" level (some can be "basic" and some "above basic," but not all 5 "above basic"). |
| Above basic | "above basic" in all 5 areas. |

- The figure used throughout most of this project is "at least basic"
  (basic and above basic combined from the table above), and it is
  conjunctive: it requires at least "basic" in *all five* areas at once,
  which means at least 5 different activities. Falling short of "basic" in
  a single area excludes someone from the figure entirely, however strong
  the other four are. The stricter "above basic" level is examined
  separately, in a closer look further down that compares it directly
  against this "at least basic" figure.
- As noted above, this proxy only runs one way, and that cuts deeper here:
  communication and information activities are part of daily life at any
  age, while content creation and problem solving are more tied to working
  and administrative life, so part of the age gap in this figure may reflect
  a change in circumstance rather than a genuine loss of skill.
- The composite figure is calculated across the whole age group, not just
  internet users: someone who did not use the internet at all in the past
  three months automatically fails every activity, which pulls the group's
  percentage down.
- The methodology changed substantially in 2021 to align with DigComp 2.0,
  so figures from 2021 onward are not comparable with anything earlier.

## Loading the data, part 1: statistics

Upload all six CSVs from `Geospatial/data/processed/` in one go. Hold Ctrl
(Cmd on Mac) in the file dialog to select all six at once:
- `household_internet_access.csv`
- `individual_internet_use.csv`
- `internet_use_by_age.csv`
- `digital_skills_by_age.csv`
- `digital_skills_by_area.csv`
- `digital_skills_above_basic.csv`

In [ ]:
from google.colab import files

uploaded_csv = files.upload()
# Select all six from data/processed/:
# - household_internet_access.csv
# - individual_internet_use.csv
# - internet_use_by_age.csv
# - digital_skills_by_age.csv
# - digital_skills_by_area.csv
# - digital_skills_above_basic.csv

print("Uploaded:", list(uploaded_csv.keys()))

## Loading the data, part 2: country boundaries

Upload the country boundary map from `Geospatial/data/geo/`:
`europe_countries_boundaries.geojson` (GISCO + OpenStreetMap for Kosovo,
Iceland and the UK; see the project README for the full provenance).

In [ ]:
uploaded_geo = files.upload()
# Select from data/geo/:
# - europe_countries_boundaries.geojson

print("Uploaded:", list(uploaded_geo.keys()))

In [ ]:
import pandas as pd
import numpy as np
import json
import plotly.graph_objects as go
from scipy.spatial import ConvexHull

with open('europe_countries_boundaries.geojson') as f:
    geojson = json.load(f)

print("Boundary features loaded:", len(geojson['features']))

## Chapter 1: How did household internet access spread across Europe, 2002–2025?

Household internet access in Europe, 2002–2025, animated by year.

- Play the animation or step through the slider.
- Hover a country for its name and coverage percentage that year.
- Amber marks a country with no data reported yet; once a country has
  reported at least once, a later gap year carries forward its last known
  shade instead of switching to amber.

Carried over from `notebooks/Digitalization.ipynb`, including the Kosovo
dedicated-trace fix (Kosovo's shape sits ~99% inside Serbia's on the GISCO
boundaries, which needs its own pair of traces to draw *and* hover correctly;
see the project README for the full story).

In [ ]:
MISSING_COLOR = 'rgb(246,217,168)'
OCEAN_COLOR = '#DFF2E8'
UNTRACKED_COLOR = 'rgb(211,209,199)'

# Play/Pause button colors, matching the map's own blue: the two shades
# swap foreground/background so the pair reads as one family. Play is the
# primary action (the whole point of the animated map), so it gets the
# stronger, dark-on-light-text version; Pause is secondary and gets the
# quieter, light-on-dark-text version. Contrast-checked at 4.21:1 (WCAG),
# comfortably above the 3:1 floor for button-sized text.
BUTTON_DARK_BLUE = '#1F5FA6'
BUTTON_LIGHT_BLUE = '#B8D4EC'

df = pd.read_csv('household_internet_access.csv')

geo_ids = {f['properties']['CNTR_ID'] for f in geojson['features']}
df_countries = df[df['geo_code'].isin(geo_ids)].sort_values('year')

all_years = range(df_countries['year'].min(), df_countries['year'].max() + 1)
all_countries = df_countries[['geo_code', 'geo_label']].drop_duplicates()

full_index = pd.MultiIndex.from_product(
    [all_countries['geo_code'], all_years], names=['geo_code', 'year']
)
df_filled = df_countries.set_index(['geo_code', 'year']).reindex(full_index).reset_index()
df_filled['geo_label'] = df_filled['geo_code'].map(all_countries.set_index('geo_code')['geo_label'])
df_filled = df_filled.sort_values(['geo_code', 'year'])
df_filled['pct_households_with_internet'] = (
    df_filled.groupby('geo_code')['pct_households_with_internet'].ffill()
)

name_lookup = all_countries.set_index('geo_code')['geo_label']
MAIN_COUNTRIES = sorted(c for c in all_countries['geo_code'] if c != 'XK')
value_lookup = df_filled.set_index(['geo_code', 'year'])['pct_households_with_internet']
years = sorted(df_filled['year'].unique())

missing_by_year = {}
for year in years:
    year_rows = df_filled[df_filled['year'] == year]
    missing = sorted(year_rows.loc[year_rows['pct_households_with_internet'].isna(), 'geo_code'])
    missing_by_year[int(year)] = missing


def country_state(code, year, missing_set):
    """(data_z, data_hover, amber_z, amber_hover) for one country - colored by
    value OR flagged amber as "no data yet", never both."""
    name = name_lookup.get(code, code)
    if code in missing_set:
        return None, "", 1, f"{name}<br>No data yet"
    v = value_lookup.get((code, year))
    return v, f"{name}<br>{v:.1f}%", None, ""


def frame_data(year):
    missing_set = set(missing_by_year.get(year, []))
    z_data, hover_data, z_amber, hover_amber = [], [], [], []
    for c in MAIN_COUNTRIES:
        dz, dh, az, ah = country_state(c, year, missing_set)
        z_data.append(dz); hover_data.append(dh)
        z_amber.append(az); hover_amber.append(ah)
    xk_dz, xk_dh, xk_az, xk_ah = country_state('XK', year, missing_set)
    return dict(
        main_data=(z_data, hover_data), main_amber=(z_amber, hover_amber),
        xk_data=([xk_dz], [xk_dh]), xk_amber=([xk_az], [xk_ah])
    )


# Small color-swatch legend for the map's background colors.
LEGEND_ITEMS = [(OCEAN_COLOR, 'Ocean'), (UNTRACKED_COLOR, 'Not tracked'), (MISSING_COLOR, 'No data yet')]
legend_shapes, legend_annotations = [], []
_start_x, _y_pos, _box_w, _gap = 0.0, 1.045, 0.018, 0.13
for i, (color, label) in enumerate(LEGEND_ITEMS):
    x0 = _start_x + i * _gap
    legend_shapes.append(dict(type='rect', xref='paper', yref='paper',
                               x0=x0, x1=x0 + _box_w, y0=_y_pos - 0.035, y1=_y_pos,
                               fillcolor=color, line=dict(color='rgba(0,0,0,0.3)', width=0.5)))
    legend_annotations.append(dict(text=label, xref='paper', yref='paper',
                                    x=x0 + _box_w + 0.006, y=_y_pos - 0.017,
                                    xanchor='left', yanchor='middle', showarrow=False, font=dict(size=10)))


def reporting_annotation(year):
    missing = missing_by_year.get(year, [])
    return dict(text=f"Not yet reporting: {', '.join(missing)}" if missing else " ",
                xref='paper', yref='paper', x=0.5, y=-0.06,
                showarrow=False, font=dict(size=11, color=MISSING_COLOR))


def traces_for(year, first=False):
    d = frame_data(year)
    geo = dict(geojson=geojson, featureidkey='properties.CNTR_ID') if first else {}
    return [
        go.Choropleth(locations=MAIN_COUNTRIES, z=d['main_data'][0], text=d['main_data'][1],
                      hoverinfo='text', colorscale='Blues', zmin=0, zmax=100,
                      marker_line_color='white', marker_line_width=0.5,
                      colorbar=dict(title='% of households<br>with internet'), **geo),
        go.Choropleth(locations=MAIN_COUNTRIES, z=d['main_amber'][0], text=d['main_amber'][1],
                      hoverinfo='text', colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]],
                      showscale=False, marker_line_color='white', marker_line_width=0.3, **geo),
        go.Choropleth(locations=['XK'], z=d['xk_data'][0], text=d['xk_data'][1],
                      hoverinfo='text', colorscale='Blues', zmin=0, zmax=100, showscale=False,
                      marker_line_color='white', marker_line_width=0.5, **geo),
        go.Choropleth(locations=['XK'], z=d['xk_amber'][0], text=d['xk_amber'][1],
                      hoverinfo='text', colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]],
                      showscale=False, marker_line_color='white', marker_line_width=0.3, **geo)
    ]


first_year = years[0]

fig = go.Figure(
    data=traces_for(first_year, first=True),
    frames=[go.Frame(name=str(year), data=traces_for(year),
                      layout=dict(annotations=legend_annotations + [reporting_annotation(year)]))
            for year in years]
)

fig.update_geos(visible=False, lonaxis_range=[-25, 45], lataxis_range=[33, 72],
                 showland=True, landcolor=UNTRACKED_COLOR, showocean=True,
                 oceancolor=OCEAN_COLOR, projection_type='equirectangular')

fig.update_layout(
    title='Household internet access in Europe, 2002-2025',
    dragmode=False, width=1000, height=700, margin=dict(t=110, b=110),
    shapes=legend_shapes, annotations=legend_annotations + [reporting_annotation(first_year)],
    sliders=[dict(active=0, currentvalue=dict(prefix='Year: '), x=0.18, len=0.80, y=-0.14, pad=dict(t=0, b=0),
                  steps=[dict(method='animate', label=str(year),
                              args=[[str(year)], dict(mode='immediate', frame=dict(duration=0, redraw=True),
                                                       transition=dict(duration=0))])
                         for year in years])],
    # Two separate updatemenus (not one menu with two buttons): plotly
    # applies bgcolor/font per MENU, not per button, so giving Play and
    # Pause different colors needs two adjacent single-button menus.
    updatemenus=[
        dict(type='buttons', direction='left', showactive=False,
             x=0.0, xanchor='left', y=-0.15, yanchor='top', pad=dict(t=0, r=10),
             bgcolor=BUTTON_DARK_BLUE, bordercolor=BUTTON_DARK_BLUE,
             font=dict(color=BUTTON_LIGHT_BLUE, size=12),
             buttons=[dict(label='Play', method='animate',
                           args=[None, dict(frame=dict(duration=400, redraw=True),
                                             fromcurrent=True, transition=dict(duration=0))])]),
        dict(type='buttons', direction='left', showactive=False,
             x=0.09, xanchor='left', y=-0.15, yanchor='top', pad=dict(t=0, r=10),
             bgcolor=BUTTON_LIGHT_BLUE, bordercolor=BUTTON_LIGHT_BLUE,
             font=dict(color=BUTTON_DARK_BLUE, size=12),
             buttons=[dict(label='Pause', method='animate',
                           args=[[None], dict(frame=dict(duration=0, redraw=False),
                                               mode='immediate', transition=dict(duration=0))])]),
    ]
)

fig1 = fig
fig1.show()


Household internet access grew significantly across the period. In 2002,
coverage ranged from 3% (Latvia) to 58% (Netherlands). By 2025, 36 countries
reported, ranging from 87.9% (Croatia) to 99.4% (Kosovo*); only five,
Croatia, Montenegro, Bosnia and Herzegovina,
Greece and Lithuania, were still below 90%.

## Chapter 2: Does coverage explain use?

Coverage and use could relate in three different ways, though the aggregate
percentages alone can't fully confirm the mechanism behind whichever
pattern shows up.

**Hypothesis 1**: use lags coverage, a household connects, but not every
member uses it actively, or some connections even sit unused for a while.

**Hypothesis 2**: use leads coverage, either people go online outside the
household, e.g. work, school, friends, internet cafes, before a connection
ever reaches home, or a newly connected household has every member using
it actively right away.

**Hypothesis 3**: the two just move together throughout, with no
consistent lead or lag in either direction.

Six looks at the relationship between household coverage and individual
use, working through which of these holds.

- The year-by-year overview across the whole 2002–2025 span.
- A country-by-country check of how closely they actually track each other.
- Whether use tends to lead coverage or follow it, and whether that changes
  over time.
- Checking that pattern against the actual country-year data.
- A closer look at two individual years side by side.
- Where things stand today.

In [ ]:
AGG_CH2 = {'EU27_2020', 'EU28', 'EU27_2007', 'EU25', 'EU15', 'EA'}
COVERAGE_BLUE_CH2 = '#1F5FA6'
USE_GRAY_CH2 = '#595959'

CHART_TITLE_CH2A = 'Comparison of household coverage and individual use in each year.'
ANSWER_CH2A = ('It looks like individual use and household coverage move closely together, '
               'rising in step across the whole period (hypothesis 3). Early on, 2002-2011, it '
               "looks more like hypothesis 2; we'll take a closer look at that further down. From "
               '2012 to 2025, it looks more like hypothesis 1 instead, coverage running slightly '
               "ahead of use on average (mean gap -1.7 points). Let's check the country-by-country "
               'picture next.')

MAJOR_TICKVALS_CH2 = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
MINOR_TICKVALS_CH2 = [5, 15, 25, 35, 45, 55, 65, 75, 85, 95]

hh = pd.read_csv('household_internet_access.csv')
iu = pd.read_csv('individual_internet_use.csv')
# A few cells carry a status flag but no value; keep those rows in the CSV
# for the flag info, but drop them here since a missing coverage or use
# figure can't be compared against anything.
hh = hh[hh['pct_households_with_internet'].notna()]
iu = iu[iu['pct_individuals_used_internet_3m'].notna()]
hh_c = hh[~hh['geo_code'].isin(AGG_CH2)]
iu_c = iu[~iu['geo_code'].isin(AGG_CH2)]
merged_ch2 = pd.merge(
    hh_c[['geo_code', 'geo_label', 'year', 'pct_households_with_internet']],
    iu_c[['geo_code', 'geo_label', 'year', 'pct_individuals_used_internet_3m']],
    on=['geo_code', 'geo_label', 'year']
)
avg_ch2 = merged_ch2.groupby('year').agg(
    coverage=('pct_households_with_internet', 'mean'),
    use=('pct_individuals_used_internet_3m', 'mean'),
).reset_index()
years_ch2a = avg_ch2['year'].tolist()

fig2a = go.Figure()
fig2a.add_trace(go.Bar(
    x=years_ch2a, y=avg_ch2['coverage'], name='Household coverage (country average)',
    marker_color=COVERAGE_BLUE_CH2,
    hovertemplate='%{x}<br>Coverage: %{y:.1f}%<extra></extra>'
))
fig2a.add_trace(go.Bar(
    x=years_ch2a, y=avg_ch2['use'], name='Individual use (country average)',
    marker_color=USE_GRAY_CH2,
    hovertemplate='%{x}<br>Use: %{y:.1f}%<extra></extra>'
))

fig2a.update_layout(
    title=f'{CHART_TITLE_CH2A}<br>'
          '<sub>Plain unweighted average across all reporting countries, no binning</sub>',
    barmode='group',
    xaxis=dict(title='Year', type='category'),
    yaxis=dict(title='%', range=[0, 105], ticksuffix='%',
                tickmode='array', tickvals=MAJOR_TICKVALS_CH2,
                minor=dict(tickmode='array', tickvals=MINOR_TICKVALS_CH2,
                           ticks='outside', ticklen=4, showgrid=False)),
    width=1000, height=650,
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.22)
)
fig2a.update_yaxes(showgrid=True, gridcolor='#EEEEEE')

fig2a.show()


It looks like individual use and household coverage move closely together,
rising in step across the whole period (hypothesis 3). Early on, 2002-2011,
it looks more like hypothesis 2; we'll take a closer look at that further
down. From 2012 to 2025, it looks more like hypothesis 1 instead, coverage
running slightly ahead of use on average (mean gap -1.7 points). Let's
check the country-by-country picture next.

### How closely do use and coverage grow together, country by country?

- **Years reported**: how many years of reported data gave inputs for the
  other two columns, out of 2002–2025.
- **Avg. gap**: the average distance between individual use and household
  coverage in that country, in percentage points, regardless of which one
  was higher.
- **Correlation**: how closely the two rose and fell together over time,
  from 0 (no relationship) to 1 (perfectly in step), independent of how far
  apart they actually sit.

In [ ]:
# Chapter 2, "closer look" inserted right after chart 1 (chapter2a_cell):
# checks, country by country, whether use and coverage really move together
# the way the country-average chart suggests. See chapter2b_country_check.py
# for the full reasoning (this is its flat, shared-namespace notebook
# mirror, `_CH2B` suffix to avoid collisions).
#
# No plotly figure here, unlike other cells: this produces a pandas
# DataFrame (shown directly, Jupyter renders it as a table) AND an HTML
# string (ch2b_table_html_CH2B) that export_cell.py's final bundle reuses
# directly, the same way it reuses figN variables for chart-based chapters.

hh_ch2b = pd.read_csv('household_internet_access.csv')
iu_ch2b = pd.read_csv('individual_internet_use.csv')
# A few cells carry a status flag but no value; keep those rows in the CSV
# for the flag info, but drop them here since a missing coverage or use
# figure can't be compared against anything.
hh_ch2b = hh_ch2b[hh_ch2b['pct_households_with_internet'].notna()]
iu_ch2b = iu_ch2b[iu_ch2b['pct_individuals_used_internet_3m'].notna()]
hh_c_ch2b = hh_ch2b[~hh_ch2b['geo_code'].isin(AGG_CH2)]
iu_c_ch2b = iu_ch2b[~iu_ch2b['geo_code'].isin(AGG_CH2)]
merged_ch2b = pd.merge(
    hh_c_ch2b[['geo_code', 'geo_label', 'year', 'pct_households_with_internet']],
    iu_c_ch2b[['geo_code', 'geo_label', 'year', 'pct_individuals_used_internet_3m']],
    on=['geo_code', 'geo_label', 'year']
)
merged_ch2b['absdiff'] = (merged_ch2b['pct_individuals_used_internet_3m']
                           - merged_ch2b['pct_households_with_internet']).abs()

rows_ch2b = []
for _, sub in merged_ch2b.groupby('geo_code'):
    label = sub['geo_label'].iloc[0]
    n = len(sub)
    mean_abs = sub['absdiff'].mean()
    corr = sub['pct_households_with_internet'].corr(sub['pct_individuals_used_internet_3m'])
    rows_ch2b.append((label, n, mean_abs, corr))

rows_ch2b.sort(key=lambda r: r[2])

country_table_CH2B = pd.DataFrame(
    rows_ch2b, columns=['Country', 'Years reported', 'Avg. gap (points)', 'Correlation']
)
country_table_CH2B['Avg. gap (points)'] = country_table_CH2B['Avg. gap (points)'].round(1)
country_table_CH2B['Correlation'] = country_table_CH2B['Correlation'].round(3)

_trs_ch2b = "\n".join(
    f'    <tr><td>{label}</td><td>{n}</td><td>{mean_abs:.1f}</td><td>{corr:.3f}</td></tr>'
    for label, n, mean_abs, corr in rows_ch2b
)
ch2b_table_html_CH2B = f"""
    <div class="table-wrap">
    <table class="stats-table">
    <tr><th>Country</th><th>Years reported</th><th>Avg. gap (points)</th><th>Correlation</th></tr>
{_trs_ch2b}
    </table>
    </div>
    """

country_table_CH2B


Use and coverage move closely together in almost every country
(hypothesis 3). Across all
749 country-year observations, the median gap is 2.7 points, and 91% are
within 10 points. Every single country correlates at 0.94 or higher, most
above 0.98. The tightest match is Switzerland (0.6 points, though only 6
years of data reported); the loosest is Albania (14.8 points), and even that
still correlates at 0.963. That closeness actually grows over time: from
2012 to 2025, the median gap narrows to 1.9 points, with 94.6% of
country-years within 10 points, both tighter than the 2002-2011 figures
(median 4.9 points, 83.6% within 10).

### A closer look at the lead-lag pattern between coverage and use

In [ ]:
# Chapter 2, closer look: testing the three coverage-vs-use hypotheses Jani
# raised. The rest of chapter 2 (2a overview, country-by-country check,
# 2b/2c detail) establishes that coverage and use track each other closely
# throughout (correlation 0.94 or higher every year). This chart answers the
# follow-up question that leaves open: which one leads, and does that change
# over time? Flat, shared-namespace notebook mirror of chapter2d_gap.py;
# reuses AGG_CH2 (chapter 2a cell). See chapter2d_gap.py for the full
# reasoning.

USE_AHEAD_COLOR_CH2D = '#2E7D5B'       # green: use ahead of coverage
COVERAGE_AHEAD_COLOR_CH2D = '#1F5FA6'  # blue: coverage ahead of use
LINE_COLOR_CH2D = '#4D4D4D'

hh_ch2d = pd.read_csv('household_internet_access.csv')
iu_ch2d = pd.read_csv('individual_internet_use.csv')
# A few cells carry a status flag but no value; keep those rows in the CSV
# for the flag info, but drop them here since a missing coverage or use
# figure can't be compared against anything.
hh_ch2d = hh_ch2d[hh_ch2d['pct_households_with_internet'].notna()]
iu_ch2d = iu_ch2d[iu_ch2d['pct_individuals_used_internet_3m'].notna()]
hh_c_ch2d = hh_ch2d[~hh_ch2d['geo_code'].isin(AGG_CH2)]
iu_c_ch2d = iu_ch2d[~iu_ch2d['geo_code'].isin(AGG_CH2)]
merged_ch2d = pd.merge(
    hh_c_ch2d[['geo_code', 'year', 'pct_households_with_internet']],
    iu_c_ch2d[['geo_code', 'year', 'pct_individuals_used_internet_3m']],
    on=['geo_code', 'year']
)
merged_ch2d['gap'] = (merged_ch2d['pct_individuals_used_internet_3m']
                       - merged_ch2d['pct_households_with_internet'])
by_year_ch2d = merged_ch2d.groupby('year')['gap'].agg(['mean', 'count']).reset_index()

marker_colors_ch2d = [USE_AHEAD_COLOR_CH2D if v >= 0 else COVERAGE_AHEAD_COLOR_CH2D
                       for v in by_year_ch2d['mean']]

fig2d = go.Figure()

# Background shading for the two eras, split at the point the mean first
# goes negative and stays negative (2012).
fig2d.add_shape(type='rect', xref='x', yref='paper', x0=2001.5, x1=2011.5, y0=0, y1=1,
                 fillcolor='rgba(46,125,91,0.06)', line=dict(width=0), layer='below')
fig2d.add_shape(type='rect', xref='x', yref='paper', x0=2011.5, x1=2025.5, y0=0, y1=1,
                 fillcolor='rgba(31,95,166,0.06)', line=dict(width=0), layer='below')

fig2d.add_hline(y=0, line=dict(color='rgba(0,0,0,0.35)', width=1, dash='dash'))

fig2d.add_trace(go.Scatter(
    x=by_year_ch2d['year'], y=by_year_ch2d['mean'], mode='lines+markers',
    line=dict(color=LINE_COLOR_CH2D, width=1.5),
    marker=dict(color=marker_colors_ch2d, size=8, line=dict(color='white', width=1)),
    text=[f"{y}: {v:+.1f} points ({n} countries)" for y, v, n in
          zip(by_year_ch2d['year'], by_year_ch2d['mean'], by_year_ch2d['count'])],
    hoverinfo='text', showlegend=False
))

fig2d.add_annotation(x=2006, y=6.9, text='<b>Mean leans toward use</b>', showarrow=False,
                      font=dict(color=USE_AHEAD_COLOR_CH2D, size=12))
fig2d.add_annotation(x=2021, y=-3.5, text='<b>Mean leans toward coverage</b>', showarrow=False,
                      font=dict(color=COVERAGE_AHEAD_COLOR_CH2D, size=12))

fig2d.update_xaxes(title='Year', dtick=2, gridcolor='#EEEEEE')
fig2d.update_yaxes(title='Mean gap (individual use minus household coverage, points)',
                    gridcolor='#EEEEEE', zeroline=False)

fig2d.update_layout(
    title=dict(text='Does individual use run ahead of household coverage, or behind it?',
               x=0.02, xanchor='left'),
    width=950, height=560, plot_bgcolor='white'
)

fig2d.show()


The mean gap flips sign right around 2012: from 2002 to about 2011, the
average country runs use ahead of coverage (mean gap +3.9 points); from
2012 on, the average flips to coverage ahead of use (mean gap -1.7
points). Correlation stays tight throughout, 0.94 or higher every year
(see the country-by-country check above), so coverage and use never drift
far apart, even as the average direction flips. That flip looks like a
clean handoff from hypothesis 2 to hypothesis 1, but averages can hide a
lot: the table below checks how many actual country-years back that up.

### Does either hypothesis actually hold up?

- **Country-year**: one country's data for one year, counted separately
  for each year it reported.
- **Hypothesis 2**: country-years where individual use was at or above
  household coverage.
- **Hypothesis 1**: country-years where household coverage was above
  individual use.

In [ ]:
# Chapter 2, closer look: checks how many country-years actually back
# hypothesis 1 or hypothesis 2 in each era, rather than trusting the mean
# gap curve (chapter2d_cell, right above) at face value. See
# chapter2e_hypothesis_table.py for the full reasoning (this is its flat,
# shared-namespace notebook mirror, `_CH2E` suffix to avoid collisions).
#
# No plotly figure here, unlike other cells: this produces a pandas
# DataFrame (shown directly, Jupyter renders it as a table) AND an HTML
# string (ch2e_table_html_CH2E) that export_cell.py's final bundle reuses
# directly, the same way it reuses figN variables for chart-based chapters.

ERAS_CH2E = [('2002-2011', 2002, 2011), ('2012-2025', 2012, 2025)]

hh_ch2e = pd.read_csv('household_internet_access.csv')
iu_ch2e = pd.read_csv('individual_internet_use.csv')
# A few cells carry a status flag but no value; keep those rows in the CSV
# for the flag info, but drop them here since a missing coverage or use
# figure can't be compared against anything.
hh_ch2e = hh_ch2e[hh_ch2e['pct_households_with_internet'].notna()]
iu_ch2e = iu_ch2e[iu_ch2e['pct_individuals_used_internet_3m'].notna()]
hh_c_ch2e = hh_ch2e[~hh_ch2e['geo_code'].isin(AGG_CH2)]
iu_c_ch2e = iu_ch2e[~iu_ch2e['geo_code'].isin(AGG_CH2)]
merged_ch2e = pd.merge(
    hh_c_ch2e[['geo_code', 'year', 'pct_households_with_internet']],
    iu_c_ch2e[['geo_code', 'year', 'pct_individuals_used_internet_3m']],
    on=['geo_code', 'year']
)
merged_ch2e['gap'] = (merged_ch2e['pct_individuals_used_internet_3m']
                       - merged_ch2e['pct_households_with_internet'])

rows_ch2e = []
for label, lo, hi in ERAS_CH2E:
    sub = merged_ch2e[(merged_ch2e['year'] >= lo) & (merged_ch2e['year'] <= hi)]
    n = len(sub)
    h2 = int((sub['gap'] >= 0).sum())
    h1 = int((sub['gap'] < 0).sum())
    rows_ch2e.append((label, n, h2, h2 / n * 100, h1, h1 / n * 100))

era_table_CH2E = pd.DataFrame(
    rows_ch2e, columns=['Period', 'Total reported country-years',
                         'Hypothesis 2 (count)', 'Hypothesis 2 (%)',
                         'Hypothesis 1 (count)', 'Hypothesis 1 (%)']
)
era_table_CH2E['Hypothesis 2 (%)'] = era_table_CH2E['Hypothesis 2 (%)'].round(1)
era_table_CH2E['Hypothesis 1 (%)'] = era_table_CH2E['Hypothesis 1 (%)'].round(1)

_trs_ch2e = "\n".join(
    f'    <tr><td>{label}</td><td>{n}</td><td>{h2} ({h2pct:.1f}%)</td>'
    f'<td>{h1} ({h1pct:.1f}%)</td></tr>'
    for label, n, h2, h2pct, h1, h1pct in rows_ch2e
)
ch2e_table_html_CH2E = f"""
    <div class="table-wrap">
    <table class="stats-table">
    <tr><th>Period</th><th>Total reported country-years</th>
    <th>Hypothesis 2 (use &ge; coverage)</th><th>Hypothesis 1 (coverage &gt; use)</th></tr>
{_trs_ch2e}
    </table>
    </div>
    """

era_table_CH2E


Neither hypothesis 1 nor hypothesis 2 is confirmed in either period, only
hypothesis 3 holds throughout. In 2002-2011, use ran ahead of coverage in a
majority of country-years (73%, hypothesis 2), but the remaining 27% still
went the other way, not decisive enough to call it confirmed. In
2012-2025, the lean reverses toward hypothesis 1 (60%), again with a large
minority, 40%, still running the other way. The mean gap chart above is
real, but it averages away roughly a quarter to two-fifths of countries
pulling in the opposite direction in every single period.

### Fifteen years of progress in coverage and use

In [ ]:
# Chapter 2, chart 2: real per-country coverage-vs-use detail for 2002 and
# 2017. Rebuilt from an earlier binned (min/max/mean per coverage-decile)
# design to plot every country's actual point directly, with a convex hull
# per year as the non-binned equivalent of the old min-max band; its
# vertices are the countries that define that year's spread, flagged
# individually. See chapter2b_binned_detail.py for the full reasoning (this
# is its flat, shared-namespace notebook mirror, `_CH2` / `_ch2b` suffixes).

EUROSTAT_TO_ISO_CH2 = {'EL': 'GR', 'UK': 'GB'}


def flag_emoji_ch2(iso2):
    if iso2 == 'XK':
        return '🏳️ XK'
    iso2 = EUROSTAT_TO_ISO_CH2.get(iso2, iso2)
    try:
        return chr(0x1F1E6 + ord(iso2[0]) - ord('A')) + chr(0x1F1E6 + ord(iso2[1]) - ord('A'))
    except Exception:
        return iso2


def bin_year_ch2(df_year):
    # No longer used by fig2b/fig2c below (rebuilt on a convex hull over the
    # real per-country points instead of coverage-decile bins), but kept
    # here because appendix_wrongtitle_cell.py and appendix_grid_cell.py
    # still call it, they're deliberately preserved as an exact record of
    # the earlier, binned design.
    bins = []
    for lo in range(0, 100, 10):
        hi = lo + 10
        if hi == 100:
            sub = df_year[(df_year['pct_households_with_internet'] >= lo) &
                          (df_year['pct_households_with_internet'] <= hi)]
        else:
            sub = df_year[(df_year['pct_households_with_internet'] >= lo) &
                          (df_year['pct_households_with_internet'] < hi)]
        if len(sub) == 0:
            continue
        min_row = sub.loc[sub['pct_individuals_used_internet_3m'].idxmin()]
        max_row = sub.loc[sub['pct_individuals_used_internet_3m'].idxmax()]
        bins.append(dict(
            lo=lo, hi=hi, mid=(lo + hi) / 2, n=len(sub),
            min_val=min_row['pct_individuals_used_internet_3m'], min_country=min_row['geo_code'],
            max_val=max_row['pct_individuals_used_internet_3m'], max_country=max_row['geo_code'],
            mean_val=sub['pct_individuals_used_internet_3m'].mean(),
        ))
    return bins


def _cluster_flags_ch2(xs, ys, codes, threshold=5.0):
    # Groups hull-vertex points within `threshold` of each other into one
    # combined flag label. Later years push several boundary countries into
    # the same near-100% corner (e.g. NL/IS/NO/SE in 2017), so placing a
    # separate flag glyph directly on each point makes them overlap into an
    # unreadable smear instead of staying legible.
    clusters = []
    for px, py, code in zip(xs, ys, codes):
        for cluster in clusters:
            cx = sum(p[0] for p in cluster) / len(cluster)
            cy = sum(p[1] for p in cluster) / len(cluster)
            if ((px - cx) ** 2 + (py - cy) ** 2) ** 0.5 <= threshold:
                cluster.append((px, py, code))
                break
        else:
            clusters.append([(px, py, code)])
    out = []
    for cluster in clusters:
        cluster.sort(key=lambda p: p[0])
        cx = sum(p[0] for p in cluster) / len(cluster)
        cy = sum(p[1] for p in cluster) / len(cluster)
        out.append((cx, cy, [p[2] for p in cluster]))
    return out


TIER_LIGHT_CH2 = dict(fill='rgba(229,229,229,0.55)', line='#767676')
TIER_DARK_CH2 = dict(fill='rgba(184,184,184,0.55)', line='#333333')
YEAR_LABEL_COLOR_CH2 = '#333333'
YEARS_CH2B = [2002, 2017]
MAJOR_TICKVALS_CH2 = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
MINOR_TICKVALS_CH2 = [5, 15, 25, 35, 45, 55, 65, 75, 85, 95]

hh = pd.read_csv('household_internet_access.csv')
iu = pd.read_csv('individual_internet_use.csv')
# A few cells carry a status flag but no value; keep those rows in the CSV
# for the flag info, but drop them here since a missing coverage or use
# figure can't be compared against anything.
hh = hh[hh['pct_households_with_internet'].notna()]
iu = iu[iu['pct_individuals_used_internet_3m'].notna()]
hh_c = hh[~hh['geo_code'].isin(AGG_CH2)]
iu_c = iu[~iu['geo_code'].isin(AGG_CH2)]
merged_ch2b = pd.merge(
    hh_c[['geo_code', 'geo_label', 'year', 'pct_households_with_internet']],
    iu_c[['geo_code', 'geo_label', 'year', 'pct_individuals_used_internet_3m']],
    on=['geo_code', 'geo_label', 'year']
)

fig2b = go.Figure()
fig2b.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines',
                            line=dict(color='rgba(0,0,0,0.3)', width=1, dash='dash'),
                            name='Reference (use = coverage)'))

year_annotations_ch2b = []
for i, year in enumerate(YEARS_CH2B):
    colors = TIER_LIGHT_CH2 if i == 0 else TIER_DARK_CH2
    df_year = merged_ch2b[merged_ch2b['year'] == year].sort_values('pct_households_with_internet')
    x_ch2b = df_year['pct_households_with_internet'].to_numpy()
    y_ch2b = df_year['pct_individuals_used_internet_3m'].to_numpy()
    codes_ch2b = df_year['geo_code'].tolist()

    fig2b.add_trace(go.Scatter(x=x_ch2b, y=y_ch2b, mode='markers',
                                marker=dict(size=6, color=colors['line'], opacity=0.55),
                                name=f'{year} countries', hoverinfo='skip'))

    pts_ch2b = np.column_stack([x_ch2b, y_ch2b])
    hull_ch2b = ConvexHull(pts_ch2b)
    hull_idx_ch2b = hull_ch2b.vertices.tolist()
    hx_ch2b = [x_ch2b[j] for j in hull_idx_ch2b] + [x_ch2b[hull_idx_ch2b[0]]]
    hy_ch2b = [y_ch2b[j] for j in hull_idx_ch2b] + [y_ch2b[hull_idx_ch2b[0]]]

    fig2b.add_trace(go.Scatter(x=hx_ch2b, y=hy_ch2b, fill='toself', fillcolor=colors['fill'],
                                line=dict(color=colors['line'], width=1.5),
                                mode='lines', name=f'{year} range (all countries)'))

    raw_x_ch2b = [x_ch2b[j] for j in hull_idx_ch2b]
    raw_y_ch2b = [y_ch2b[j] for j in hull_idx_ch2b]
    raw_codes_ch2b = [codes_ch2b[j] for j in hull_idx_ch2b]
    clustered_ch2b = _cluster_flags_ch2(raw_x_ch2b, raw_y_ch2b, raw_codes_ch2b)
    flag_x_ch2b = [c[0] for c in clustered_ch2b]
    flag_y_ch2b = [c[1] for c in clustered_ch2b]
    flag_text_ch2b = ['<b>' + ' '.join(flag_emoji_ch2(code) for code in c[2]) + '</b>' for c in clustered_ch2b]
    flag_pos_ch2b = ['middle left' if cx > 90 else 'top center' for cx, cy, _ in clustered_ch2b]
    fig2b.add_trace(go.Scatter(x=flag_x_ch2b, y=flag_y_ch2b, mode='text', text=flag_text_ch2b,
                                textposition=flag_pos_ch2b, textfont=dict(size=14, color='#1F5FA6'),
                                showlegend=False, hoverinfo='skip'))

    label_x = float(np.mean(x_ch2b))
    label_y = float(np.max(y_ch2b)) + 10
    year_annotations_ch2b.append(dict(
        x=label_x, y=label_y, xref='x', yref='y', showarrow=False,
        text=f'<b>{year}</b>', font=dict(size=15, color=colors['line'])
    ))

fig2b.update_xaxes(title='Household coverage (%)', range=[0, 105],
                    tickmode='array', tickvals=MAJOR_TICKVALS_CH2,
                    minor=dict(tickmode='array', tickvals=MINOR_TICKVALS_CH2,
                               ticks='outside', ticklen=4, showgrid=False))
fig2b.update_yaxes(title='Individual use (%)', range=[-5, 118],
                    tickmode='array', tickvals=MAJOR_TICKVALS_CH2,
                    minor=dict(tickmode='array', tickvals=MINOR_TICKVALS_CH2,
                               ticks='outside', ticklen=4, showgrid=False))

bullets_ch2b = [
    "Each shaded area spans every reporting country's actual coverage and "
    'use that year; labels mark the countries at its edges.',
]
bullet_html_ch2b = '<br>'.join(bullets_ch2b)
fig2b.update_layout(
    title=dict(text=f'Household coverage and individual use, all countries, 2002 and 2017<br><sub>{bullet_html_ch2b}</sub>',
               x=0.02, xanchor='left'),
    width=1000, height=800,
    # Extra top margin opens up clearance above the plot; legend.y stays
    # close to the plot top (1.02) since legend y is relative to the plot
    # area's top edge, a bigger y value moves the legend closer to the
    # title/subtitle block, not further from it.
    margin=dict(t=175, b=90),
    plot_bgcolor='white',
    annotations=year_annotations_ch2b,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
)

fig2b.show()


In 2002, individual use was as low as 14.7% in Greece, while household
coverage lagged even further behind: from just 3.0% in Latvia to a high of
only 58.0% in the Netherlands, no country had reached even 60% household
coverage yet. Fifteen years later, every reporting country had crossed the
60% mark on both fronts: the lowest use, 63.4% in Bulgaria, and the lowest
coverage, 67.3%, also Bulgaria.

### One more year: 2025

In [ ]:

# Chapter 2, chart 3: the present day, 2025 alone. Reuses chapter2b's
# flag_emoji_ch2 / _cluster_flags_ch2 / merged_ch2b / TIER_LIGHT_CH2 /
# TIER_DARK_CH2 / MAJOR_TICKVALS_CH2 / MINOR_TICKVALS_CH2 defined in the
# chapter 2b cell above; same convex-hull rebuild as chapter2b_cell.py. Its
# own bullets_ch2c (singular "The shaded area", not "Each"), since this
# chart only ever has one shaded area (a single year), unlike chapter2b's
# two.

YEARS_CH2C = [2025]

fig2c = go.Figure()
fig2c.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines',
                            line=dict(color='rgba(0,0,0,0.3)', width=1, dash='dash'),
                            name='Reference (use = coverage)'))

year_annotations_ch2c = []
for i, year in enumerate(YEARS_CH2C):
    colors = TIER_LIGHT_CH2 if i == 0 else TIER_DARK_CH2
    df_year = merged_ch2b[merged_ch2b['year'] == year].sort_values('pct_households_with_internet')
    x_ch2c = df_year['pct_households_with_internet'].to_numpy()
    y_ch2c = df_year['pct_individuals_used_internet_3m'].to_numpy()
    codes_ch2c = df_year['geo_code'].tolist()

    fig2c.add_trace(go.Scatter(x=x_ch2c, y=y_ch2c, mode='markers',
                                marker=dict(size=6, color=colors['line'], opacity=0.55),
                                name=f'{year} countries', hoverinfo='skip'))

    pts_ch2c = np.column_stack([x_ch2c, y_ch2c])
    hull_ch2c = ConvexHull(pts_ch2c)
    hull_idx_ch2c = hull_ch2c.vertices.tolist()
    hx_ch2c = [x_ch2c[j] for j in hull_idx_ch2c] + [x_ch2c[hull_idx_ch2c[0]]]
    hy_ch2c = [y_ch2c[j] for j in hull_idx_ch2c] + [y_ch2c[hull_idx_ch2c[0]]]

    fig2c.add_trace(go.Scatter(x=hx_ch2c, y=hy_ch2c, fill='toself', fillcolor=colors['fill'],
                                line=dict(color=colors['line'], width=1.5),
                                mode='lines', name=f'{year} range (all countries)'))

    raw_x_ch2c = [x_ch2c[j] for j in hull_idx_ch2c]
    raw_y_ch2c = [y_ch2c[j] for j in hull_idx_ch2c]
    raw_codes_ch2c = [codes_ch2c[j] for j in hull_idx_ch2c]
    clustered_ch2c = _cluster_flags_ch2(raw_x_ch2c, raw_y_ch2c, raw_codes_ch2c)
    flag_x_ch2c = [c[0] for c in clustered_ch2c]
    flag_y_ch2c = [c[1] for c in clustered_ch2c]
    flag_text_ch2c = ['<b>' + ' '.join(flag_emoji_ch2(code) for code in c[2]) + '</b>' for c in clustered_ch2c]
    flag_pos_ch2c = ['middle left' if cx > 90 else 'top center' for cx, cy, _ in clustered_ch2c]
    fig2c.add_trace(go.Scatter(x=flag_x_ch2c, y=flag_y_ch2c, mode='text', text=flag_text_ch2c,
                                textposition=flag_pos_ch2c, textfont=dict(size=14, color='#1F5FA6'),
                                showlegend=False, hoverinfo='skip'))

    label_x = float(np.mean(x_ch2c))
    label_y = float(np.max(y_ch2c)) + 10
    year_annotations_ch2c.append(dict(
        x=label_x, y=label_y, xref='x', yref='y', showarrow=False,
        text=f'<b>{year}</b>', font=dict(size=15, color=colors['line'])
    ))

fig2c.update_xaxes(title='Household coverage (%)', range=[0, 105],
                    tickmode='array', tickvals=MAJOR_TICKVALS_CH2,
                    minor=dict(tickmode='array', tickvals=MINOR_TICKVALS_CH2,
                               ticks='outside', ticklen=4, showgrid=False))
fig2c.update_yaxes(title='Individual use (%)', range=[-5, 118],
                    tickmode='array', tickvals=MAJOR_TICKVALS_CH2,
                    minor=dict(tickmode='array', tickvals=MINOR_TICKVALS_CH2,
                               ticks='outside', ticklen=4, showgrid=False))

bullets_ch2c = [
    "The shaded area spans every reporting country's actual coverage and "
    'use that year; labels mark the countries at its edges.',
]
bullet_html_ch2c = '<br>'.join(bullets_ch2c)
fig2c.update_layout(
    title=dict(text=f'And what does the present look like?<br><sub>{bullet_html_ch2c}</sub>',
               x=0.02, xanchor='left'),
    width=1000, height=800,
    # Extra top margin opens up clearance above the plot; legend.y stays
    # close to the plot top (1.02) since legend y is relative to the plot
    # area's top edge, a bigger y value moves the legend closer to the
    # title/subtitle block, not further from it.
    margin=dict(t=175, b=90),
    plot_bgcolor='white',
    annotations=year_annotations_ch2c,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
)

fig2c.show()


By 2025, every country was above 85% individual use. The lowest, Croatia,
stood at 85.9%, just 14 points behind the highest, Ireland at 99.8%.
Hypothesis 1 holds for most countries: in 21 of 36, coverage runs slightly
ahead of use. The other 15 still follow hypothesis 2, use at or above
coverage, Montenegro, Ireland, Sweden, Denmark and Bosnia and Herzegovina
lead that group.

*Along the way we also tried a straight correlation scatter and a 9-panel
year-by-year grid before landing on this three-chart format; see the
appendix at the end of this notebook for those.*

## Chapter 3: How much does internet use differ among generations?

Household coverage alongside individual internet use by age group, EU27,
2021–2025.

- Household coverage in blue; individual use by age group in green,
  lightest shade is youngest.
- EU27 here is Eurostat's own published aggregate, calculated from pooled
  microdata across member states, which effectively weights by each
  country's population.
- The age breakdown only goes back to 2021, a shorter window than the map
  in chapter 1.

In [ ]:
COVERAGE_BLUE = '#1F5FA6'
AGE_GREENS = {
    '16-24': '#A9DFB0',   # lightest = youngest
    '25-54': '#4FAE64',
    '55-74': '#1A7A47',   # darkest = oldest (fixed shade from prior session)
}
YEARS_CH3 = [2021, 2022, 2023, 2024, 2025]
EU27 = 'EU27_2020'

hh = pd.read_csv('household_internet_access.csv')
iu_age = pd.read_csv('internet_use_by_age.csv')

coverage = (
    hh[(hh['geo_code'] == EU27) & (hh['year'].isin(YEARS_CH3))]
    .set_index('year')['pct_households_with_internet']
)

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=YEARS_CH3, y=[coverage[y] for y in YEARS_CH3],
    name='Household coverage', marker_color=COVERAGE_BLUE,
    text=[f'{coverage[y]:.1f}%' for y in YEARS_CH3],
    textposition='inside', textfont=dict(color='white', size=12, family='Arial Black'),
    hovertemplate='Household coverage<br>%{x}: %{y:.1f}%<extra></extra>'
))

# In-bar label color: two colors, not three. 16-24 (lightest fill) gets
# the darkest green from the family; 25-54 and 55-74 both use white
# instead of a third shade (contrast-checked, see chapter3_age_bars.py).
AGE_LABEL_TEXT_COLOR_CH3 = {
    '16-24': AGE_GREENS['55-74'],
    '25-54': 'white',
    '55-74': 'white',
}

for age in ['16-24', '25-54', '55-74']:
    sub = (
        iu_age[(iu_age['geo_code'] == EU27) & (iu_age['age_group'] == age) & (iu_age['year'].isin(YEARS_CH3))]
        .set_index('year')['pct_individuals_used_internet_3m']
    )
    fig3.add_trace(go.Bar(
        x=YEARS_CH3, y=[sub[y] for y in YEARS_CH3],
        name=f'Individual use, {age}', marker_color=AGE_GREENS[age],
        text=[f'{sub[y]:.1f}%' for y in YEARS_CH3],
        textposition='inside', textfont=dict(color=AGE_LABEL_TEXT_COLOR_CH3[age], size=12, family='Arial Black'),
        hovertemplate=f'Individual use, {age}<br>' + '%{x}: %{y:.1f}%<extra></extra>'
    ))

fig3.update_layout(
    title='Internet coverage and use by age group, EU27, 2021–2025',
    barmode='group',
    xaxis=dict(title='Year', type='category'),
    yaxis=dict(title='%', range=[0, 105], ticksuffix='%'),
    width=900, height=600,
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.25)
)
fig3.update_yaxes(showgrid=True, gridcolor='#EEEEEE')

fig3.show()


The age gap in internet use is closing, driven almost entirely by the
oldest group. Between 2021 and 2025 the gap between 16-24 and 55-74 use
narrowed from 21.8 points to 12.6 points, as the 55-74 group rose from
75.9% to 86.2% while the two younger groups were already close to
saturated.

## Chapter 4: How does performance of digital activities differ across generations?

As a reminder: Eurostat calls this indicator "digital skills," but this
project uses "digital activities" throughout instead, since respondents are
only asked yes/no whether they performed specific activities, never tested
(see "About the data" above).

- Shows the composite indicator: basic-or-above in *all five* DigComp
  areas at once (information & data literacy, communication & collaboration,
  digital content creation, safety, problem solving).
- Click a year/age-group combination below the map.
- Falling short in a single area excludes someone from this figure entirely,
  which is the main reason the 55–74 numbers look so low.

Carried over from `notebooks/DigitalSkills.ipynb`.

In [ ]:
df_raw = pd.read_csv('digital_skills_by_age.csv')

MISSING_COLOR = 'rgb(246,217,168)'
OCEAN_COLOR = 'rgb(223,242,232)'
UNTRACKED_COLOR = 'rgb(211,209,199)'

# Button colors: one uniform light purple background with dark purple labels
# for every unselected button in the 4x3 age-group x year grid, matching the
# map's own Purples colorscale. Contrast-checked at 7.47:1 (WCAG),
# comfortably clear of the 4.5:1 text floor. The selected button reverses
# the same pair (dark purple background, light purple label).
BUTTON_BG_PURPLE_CH4 = '#E5D8F0'
BUTTON_TEXT_PURPLE_CH4 = '#54278F'
BUTTON_BG_ACTIVE_CH4 = BUTTON_TEXT_PURPLE_CH4
BUTTON_TEXT_ACTIVE_CH4 = BUTTON_BG_PURPLE_CH4


def _pad_button_labels_ch4(labels):
    """Center-pad every label to the length of the longest one, so buttons
    in the same grid render at close to the same width with even gaps."""
    width = max(len(s) for s in labels)
    return [s.center(width) for s in labels]


AGE_GROUPS = ['16-24', '25-54', '55-74', 'All individuals (16-74)']
AGE_LABELS = {'All individuals (16-74)': 'All (16-74)', '16-24': '16-24', '25-54': '25-54', '55-74': '55-74'}
YEARS = sorted(df_raw['year'].unique())
YEAR_LABELS_PADDED_CH4 = dict(zip(YEARS, _pad_button_labels_ch4([str(y) for y in YEARS])))

geo_ids = {f['properties']['CNTR_ID'] for f in geojson['features']}
# Kosovo is deliberately EXCLUDED here and given its own pair of traces
# below - see the README / Digitalization.ipynb for the full reasoning
# (its shape sits ~99% inside Serbia's on the GISCO boundaries).
countries_age = sorted(c for c in geo_ids if c != 'XK')
country_name = {f['properties']['CNTR_ID']: f['properties']['NAME_ENGL'] for f in geojson['features']}

combo_data = {}
for year in YEARS:
    for age in AGE_GROUPS:
        subset = df_raw[(df_raw['year'] == year) & (df_raw['age_group'] == age)]
        values = subset.set_index('geo_code')['pct_basic_or_above_digital_skills']
        z = [values.get(c) for c in countries_age]
        hover = [
            f"{country_name.get(c, c)}<br>{v:.1f}%" if pd.notna(v) else f"{country_name.get(c, c)}<br>No data"
            for c, v in zip(countries_age, z)
        ]
        missing_z = [1 if pd.isna(v) else None for v in z]
        missing_hover = [
            f"{country_name.get(c, c)}<br>No data yet" if pd.isna(v) else ""
            for c, v in zip(countries_age, z)
        ]
        xk_v = values.get('XK')
        xk_has = pd.notna(xk_v)
        combo_data[(year, age)] = {
            'z': z, 'hover': hover, 'missing_z': missing_z, 'missing_hover': missing_hover,
            'xk_z': [xk_v if xk_has else None],
            'xk_hover': [f"{country_name.get('XK', 'XK')}<br>{xk_v:.1f}%" if xk_has else ""],
            'xk_missing_z': [None if xk_has else 1],
            'xk_missing_hover': ["" if xk_has else f"{country_name.get('XK', 'XK')}<br>No data yet"]
        }

# Defaults to the most recent year, since the answer text below only
# discusses 2025 figures.
default_year, default_age = YEARS[-1], 'All individuals (16-74)'
default = combo_data[(default_year, default_age)]

fig_age = go.Figure()
fig_age.add_trace(go.Choropleth(
    locations=countries_age, z=default['z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale='Purples', zmin=0, zmax=100, text=default['hover'], hoverinfo='text',
    marker_line_color='white', marker_line_width=0.5,
    colorbar=dict(title='% with basic+<br>digital activities')
))
fig_age.add_trace(go.Choropleth(
    locations=countries_age, z=default['missing_z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]], showscale=False,
    text=default['missing_hover'], hoverinfo='text', marker_line_color='white', marker_line_width=0.3
))
fig_age.add_trace(go.Choropleth(
    locations=['XK'], z=default['xk_z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale='Purples', zmin=0, zmax=100, showscale=False,
    text=default['xk_hover'], hoverinfo='text', marker_line_color='white', marker_line_width=0.5
))
fig_age.add_trace(go.Choropleth(
    locations=['XK'], z=default['xk_missing_z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]], showscale=False,
    text=default['xk_missing_hover'], hoverinfo='text', marker_line_color='white', marker_line_width=0.3
))

fig_age.update_geos(visible=False, lonaxis_range=[-25, 45], lataxis_range=[33, 72],
                    showland=True, landcolor=UNTRACKED_COLOR, showocean=True,
                    oceancolor=OCEAN_COLOR, projection_type='equirectangular',
                    domain=dict(y=[0.30, 1]))

# No chart title: an update-on-click title triggers a Plotly.js relayout
# bug where it vanishes permanently after the first button click. The
# button grid already makes the selected age/year unambiguous.
fig_age.update_layout(
    width=1050, height=920, dragmode=False
)

LEGEND_ITEMS = [(OCEAN_COLOR, 'Ocean'), (UNTRACKED_COLOR, 'Not tracked'), (MISSING_COLOR, 'No data yet')]
legend_shapes, legend_annotations = [], []
_start_x, _y_pos, _box_w, _gap = 0.02, 0.995, 0.018, 0.13
for i, (color, label) in enumerate(LEGEND_ITEMS):
    x0 = _start_x + i * _gap
    legend_shapes.append(dict(type='rect', xref='paper', yref='paper',
                               x0=x0, x1=x0 + _box_w, y0=_y_pos - 0.035, y1=_y_pos,
                               fillcolor=color, line=dict(color='rgba(0,0,0,0.3)', width=0.5)))
    legend_annotations.append(dict(text=label, xref='paper', yref='paper',
                                    x=x0 + _box_w + 0.006, y=_y_pos - 0.017,
                                    xanchor='left', yanchor='middle', showarrow=False, font=dict(size=10)))

# Plotly has no per-button color attribute (bgcolor/font only apply at the
# whole-menu level), so to give the SELECTED button its own reversed color
# each button gets its own single-button menu, and every menu's bgcolor/font
# gets flipped via the click args, the same trick already used below to keep
# only one row highlighted at a time ('updatemenus[other_row].active': -1).
# Rows = age group (4, youngest to oldest, "All" last), columns = year (3):
# same row axis as chapter 5c, so the two maps share one system.
row_y_positions_ch4 = [0.22, 0.155, 0.09, 0.025]
col_x_positions_ch4 = [0.18, 0.36, 0.54]
combos_ch4 = [(row_i, col_i) for row_i in range(len(AGE_GROUPS)) for col_i in range(len(YEARS))]


def menu_index_ch4(row_i, col_i):
    return row_i * len(YEARS) + col_i


default_row_ch4 = AGE_GROUPS.index(default_age)
default_col_ch4 = YEARS.index(default_year)

updatemenus = []
row_labels = []
for row_i, age in enumerate(AGE_GROUPS):
    for col_i, year in enumerate(YEARS):
        d = combo_data[(year, age)]
        is_default_active = (row_i == default_row_ch4 and col_i == default_col_ch4)
        layout_update = {}
        for other_row, other_col in combos_ch4:
            idx = menu_index_ch4(other_row, other_col)
            selected = (other_row == row_i and other_col == col_i)
            layout_update[f'updatemenus[{idx}].bgcolor'] = BUTTON_BG_ACTIVE_CH4 if selected else BUTTON_BG_PURPLE_CH4
            layout_update[f'updatemenus[{idx}].bordercolor'] = BUTTON_BG_ACTIVE_CH4 if selected else BUTTON_BG_PURPLE_CH4
            layout_update[f'updatemenus[{idx}].font.color'] = BUTTON_TEXT_ACTIVE_CH4 if selected else BUTTON_TEXT_PURPLE_CH4
        updatemenus.append(dict(
            type='buttons', direction='right', showactive=False,
            buttons=[dict(
                label=YEAR_LABELS_PADDED_CH4[year], method='update',
                args=[{'z': [d['z'], d['missing_z'], d['xk_z'], d['xk_missing_z']],
                       'text': [d['hover'], d['missing_hover'], d['xk_hover'], d['xk_missing_hover']]},
                      layout_update]
            )],
            x=col_x_positions_ch4[col_i], xanchor='left', y=row_y_positions_ch4[row_i], yanchor='middle',
            pad=dict(t=2, r=2, b=2, l=2),
            bgcolor=BUTTON_BG_ACTIVE_CH4 if is_default_active else BUTTON_BG_PURPLE_CH4,
            bordercolor=BUTTON_BG_ACTIVE_CH4 if is_default_active else BUTTON_BG_PURPLE_CH4,
            font=dict(color=BUTTON_TEXT_ACTIVE_CH4 if is_default_active else BUTTON_TEXT_PURPLE_CH4, size=11)
        ))
    row_labels.append(dict(text=f"<b>{AGE_LABELS[age]}</b>", xref='paper', yref='paper',
                            x=0.08, y=row_y_positions_ch4[row_i], xanchor='right', yanchor='middle',
                            showarrow=False, font=dict(size=12)))

fig_age.update_layout(updatemenus=updatemenus, annotations=row_labels + legend_annotations, shapes=legend_shapes)

fig4 = fig_age
fig4.show()


Performance on this five-activity composite does not track plain
internet use very closely, and the gap widens with age. The composite only
counts someone once they clear all five activity areas, so it runs well
below internet use at every age. In 2025, EU27 internet use
exceeded it by 24.3 points for 16-24
(98.8% vs. 74.6%), 29.0 points for 25-54 (97.5% vs. 68.6%), and 43.6 points
for 55-74 (86.2% vs. 42.6%). Being online is close to universal at every
age; clearing all five activity areas at once is not.

## Chapter 5: Which single area explains most of the generational gap?

- Breaking the composite figure down by individual area, EU27, 2025.
- Areas are shown separately from the conjunctive "Overall" figure (right of
  the dotted line), which needs all five areas at once, not one on its own.

*Recoloring each area into its own red/coral shade (instead of the current
age-based green) was discussed and deliberately deferred: the age
comparison is more central to the story than an area-by-area color key, so
this chart keeps the green age scheme for now.*

In [ ]:
AGE_GREENS = {
    '16-24': '#A9DFB0',
    '25-54': '#4FAE64',
    '55-74': '#1A7A47',
}
AREA_ORDER = [
    'Communication and collaboration',
    'Information and data literacy',
    'Problem solving',
    'Safety',
    'Digital content creation',
    'Overall (all five areas)',
]
EU27 = 'EU27_2020'

d = pd.read_csv('digital_skills_by_area.csv')
sub = d[(d['geo_code'] == EU27) & (d['year'] == 2025)]

fig5a = go.Figure()
for age in ['16-24', '25-54', '55-74']:
    s = sub[sub['age_group'] == age].set_index('skill_area')['pct_basic_or_above']
    y_vals = [s.get(area) for area in AREA_ORDER]
    fig5a.add_trace(go.Bar(
        x=AREA_ORDER, y=y_vals,
        name=age, marker_color=AGE_GREENS[age],
        text=[f'{v:.0f}%' if v is not None else '' for v in y_vals],
        textposition='outside',
        hovertemplate=f'{age}<br>' + '%{x}: %{y:.1f}%<extra></extra>'
    ))

# Visually separate the conjunctive "Overall" bar from the five individual
# areas, since it isn't measuring the same thing (all five at once, not
# one area).
fig5a.add_vline(x=4.5, line=dict(color='rgba(0,0,0,0.25)', dash='dot', width=1))

fig5a.update_layout(
    title="Digital activities by area, EU27 2025<br><sub>The generational gap looks smaller per area than in the headline figure</sub>",
    barmode='group',
    xaxis=dict(title=None, tickangle=-15),
    yaxis=dict(title='% with basic+ activities', range=[0, 108], ticksuffix='%'),
    width=1050, height=650,
    plot_bgcolor='white',
    legend=dict(title='Age group', orientation='h', yanchor='bottom', y=-0.32)
)
fig5a.update_yaxes(showgrid=True, gridcolor='#EEEEEE')

fig5a.show()


Digital content creation, by a wide margin: a 36.7-point gap between
16-24 and 55-74 (88.5% vs. 51.8%), more than double the narrowest gap,
communication and collaboration, at 15.5 points (98.4% vs. 82.9%).
Communication and collaboration is in fact the most common activity for
every age group, and content creation is also the least common one for
both 25-54 and 55-74 (78.3% and 51.8%). The youngest group's own weak spot
is different: safety (83.8%), even though that's still the highest safety
score of the three groups, their content-creation score (88.5%) is simply
even higher. In fact, 9.2 percentage points of all seniors clear digital
content creation on its own but fail to clear at least one of the other
four areas, part of why the overall composite figure sits at only 42.6%.
As "About the data" above notes, this only measures having done the
activity recently, not being able to; a 55-74 year old who hasn't opened a
spreadsheet since retiring scores the same as someone who never learned
how, so part of this gap may reflect a change in circumstance.

### Where each activity stands across countries, right now?

- A country-level view of one skill area at a time, 2025 only (the only
  year this dataset has).
- Pick an age group and an activity area below the map.
- Purple, matching chapter 4: this is a closer look at the same
  measurement, not a different one.

**Even within a single well-defined activity and a single age group, how
much do countries vary?**

In [ ]:
# Complements chapter 4 (composite, all-five-at-once) and chapter 5a/5b
# (EU27 aggregate only): the first country-level view of an INDIVIDUAL
# DigComp area. digital_skills_by_area.csv only has one year (2025), so no
# year dimension here, only skill area x age group. Rows = age group (short
# labels, youngest to oldest, "All individuals" last - same row axis as
# chapter 4), columns = skill area (given the map's full width to spread
# across), same per-button-menu color trick as chapter 4. Purple, matching
# chapter 4's composite map: same topic, deeper slice, not a new metric.
# Button labels are center-padded to equal length so every button in the
# grid renders at close to the same width with even gaps between them.
#
# Default view: Digital content creation x 55-74, the single widest
# area/age-group spread in the whole dataset (9.4% Albania - 92.0%
# Kosovo*, 82.7 points) - the sharpest illustration of "access is solved,
# activity is not, and that's a country story as much as a generational
# one."

MISSING_COLOR = 'rgb(246,217,168)'
OCEAN_COLOR = 'rgb(223,242,232)'
UNTRACKED_COLOR = 'rgb(211,209,199)'

BUTTON_BG_PURPLE_5C = '#E5D8F0'
BUTTON_TEXT_PURPLE_5C = '#54278F'
BUTTON_BG_ACTIVE_5C = BUTTON_TEXT_PURPLE_5C
BUTTON_TEXT_ACTIVE_5C = BUTTON_BG_PURPLE_5C

AREA_ORDER_5C = [
    'Digital content creation',
    'Safety',
    'Problem solving',
    'Information and data literacy',
    'Communication and collaboration',
]
AREA_LABELS_5C = {
    'Digital content creation': 'Content creation',
    'Safety': 'Safety',
    'Problem solving': 'Problem solving',
    'Information and data literacy': 'Info literacy',
    'Communication and collaboration': 'Communication',
}
AGE_GROUPS_5C = ['16-24', '25-54', '55-74', 'All individuals (16-74)']
AGE_LABELS_5C = {'All individuals (16-74)': 'All (16-74)', '16-24': '16-24', '25-54': '25-54', '55-74': '55-74'}


def _pad_button_labels_5c(labels):
    """Center-pad every label to the length of the longest one, so buttons
    in the same grid render at close to the same width with even gaps."""
    width = max(len(s) for s in labels)
    return [s.center(width) for s in labels]


AREA_LABELS_PADDED_5C = dict(zip(AREA_ORDER_5C, _pad_button_labels_5c([AREA_LABELS_5C[a] for a in AREA_ORDER_5C])))

QUESTION_CH5C = 'Even within a single well-defined activity and a single age group, how much do countries vary?'
ANSWER_CH5C = ('Enormously, far more than access itself. Among 55-74 year olds, digital content '
               'creation ranges from 9.4% (Albania) to 92.0% (Kosovo*), an 82.7-point spread, the '
               'widest of any area/age-group combination in the data, in a set of countries that '
               'mostly all have near-universal household internet access by now.')

d = pd.read_csv('digital_skills_by_area.csv')
d = d[d['skill_area'].isin(AREA_ORDER_5C)]

geo_ids = {f['properties']['CNTR_ID'] for f in geojson['features']}
countries_5c = sorted(c for c in geo_ids if c != 'XK')
country_name_5c = {f['properties']['CNTR_ID']: f['properties']['NAME_ENGL'] for f in geojson['features']}

combo_data = {}
for area in AREA_ORDER_5C:
    for age in AGE_GROUPS_5C:
        subset = d[(d['skill_area'] == area) & (d['age_group'] == age)]
        values = subset.set_index('geo_code')['pct_basic_or_above']
        z = [values.get(c) for c in countries_5c]
        hover = [
            f"{country_name_5c.get(c, c)}<br>{v:.1f}%" if pd.notna(v) else f"{country_name_5c.get(c, c)}<br>No data"
            for c, v in zip(countries_5c, z)
        ]
        missing_z = [1 if pd.isna(v) else None for v in z]
        missing_hover = [
            f"{country_name_5c.get(c, c)}<br>No data yet" if pd.isna(v) else ""
            for c, v in zip(countries_5c, z)
        ]
        xk_v = values.get('XK')
        xk_has = pd.notna(xk_v)
        combo_data[(area, age)] = {
            'z': z, 'hover': hover, 'missing_z': missing_z, 'missing_hover': missing_hover,
            'xk_z': [xk_v if xk_has else None],
            'xk_hover': [f"{country_name_5c.get('XK', 'XK')}<br>{xk_v:.1f}%" if xk_has else ""],
            'xk_missing_z': [None if xk_has else 1],
            'xk_missing_hover': ["" if xk_has else f"{country_name_5c.get('XK', 'XK')}<br>No data yet"]
        }

default_area_5c, default_age_5c = 'Digital content creation', '55-74'
default = combo_data[(default_area_5c, default_age_5c)]

fig5c = go.Figure()
fig5c.add_trace(go.Choropleth(
    locations=countries_5c, z=default['z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale='Purples', zmin=0, zmax=100, text=default['hover'], hoverinfo='text',
    marker_line_color='white', marker_line_width=0.5,
    colorbar=dict(title='% with basic+<br>in this area')
))
fig5c.add_trace(go.Choropleth(
    locations=countries_5c, z=default['missing_z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]], showscale=False,
    text=default['missing_hover'], hoverinfo='text', marker_line_color='white', marker_line_width=0.3
))
fig5c.add_trace(go.Choropleth(
    locations=['XK'], z=default['xk_z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale='Purples', zmin=0, zmax=100, showscale=False,
    text=default['xk_hover'], hoverinfo='text', marker_line_color='white', marker_line_width=0.5
))
fig5c.add_trace(go.Choropleth(
    locations=['XK'], z=default['xk_missing_z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]], showscale=False,
    text=default['xk_missing_hover'], hoverinfo='text', marker_line_color='white', marker_line_width=0.3
))

fig5c.update_geos(visible=False, lonaxis_range=[-25, 45], lataxis_range=[33, 72],
                   showland=True, landcolor=UNTRACKED_COLOR, showocean=True,
                   oceancolor=OCEAN_COLOR, projection_type='equirectangular',
                   domain=dict(y=[0.30, 1]))

# No chart title: an update-on-click title triggers a Plotly.js relayout
# bug where it vanishes permanently after the first button click. The
# button grid already makes the selected area/age unambiguous.
fig5c.update_layout(
    width=1050, height=920, dragmode=False
)

LEGEND_ITEMS = [(OCEAN_COLOR, 'Ocean'), (UNTRACKED_COLOR, 'Not tracked'), (MISSING_COLOR, 'No data yet')]
legend_shapes, legend_annotations = [], []
_start_x, _y_pos, _box_w, _gap = 0.02, 0.995, 0.018, 0.13
for i, (color, label) in enumerate(LEGEND_ITEMS):
    x0 = _start_x + i * _gap
    legend_shapes.append(dict(type='rect', xref='paper', yref='paper',
                               x0=x0, x1=x0 + _box_w, y0=_y_pos - 0.035, y1=_y_pos,
                               fillcolor=color, line=dict(color='rgba(0,0,0,0.3)', width=0.5)))
    legend_annotations.append(dict(text=label, xref='paper', yref='paper',
                                    x=x0 + _box_w + 0.006, y=_y_pos - 0.017,
                                    xanchor='left', yanchor='middle', showarrow=False, font=dict(size=10)))

row_y_positions_5c = [0.22, 0.155, 0.09, 0.025]
col_x_positions_5c = [0.14, 0.31, 0.48, 0.65, 0.82]

combos_5c = [(row_i, col_i) for row_i in range(len(AGE_GROUPS_5C)) for col_i in range(len(AREA_ORDER_5C))]


def menu_index_5c(row_i, col_i):
    return row_i * len(AREA_ORDER_5C) + col_i


updatemenus = []
row_labels = []
default_row_5c = AGE_GROUPS_5C.index(default_age_5c)
default_col_5c = AREA_ORDER_5C.index(default_area_5c)
for row_i, age in enumerate(AGE_GROUPS_5C):
    for col_i, area in enumerate(AREA_ORDER_5C):
        dd = combo_data[(area, age)]
        is_default_active = (row_i == default_row_5c and col_i == default_col_5c)
        layout_update = {}
        for other_row, other_col in combos_5c:
            idx = menu_index_5c(other_row, other_col)
            selected = (other_row == row_i and other_col == col_i)
            layout_update[f'updatemenus[{idx}].bgcolor'] = BUTTON_BG_ACTIVE_5C if selected else BUTTON_BG_PURPLE_5C
            layout_update[f'updatemenus[{idx}].bordercolor'] = BUTTON_BG_ACTIVE_5C if selected else BUTTON_BG_PURPLE_5C
            layout_update[f'updatemenus[{idx}].font.color'] = BUTTON_TEXT_ACTIVE_5C if selected else BUTTON_TEXT_PURPLE_5C
        updatemenus.append(dict(
            type='buttons', direction='right', showactive=False,
            buttons=[dict(
                label=AREA_LABELS_PADDED_5C[area], method='update',
                args=[{'z': [dd['z'], dd['missing_z'], dd['xk_z'], dd['xk_missing_z']],
                       'text': [dd['hover'], dd['missing_hover'], dd['xk_hover'], dd['xk_missing_hover']]},
                      layout_update]
            )],
            x=col_x_positions_5c[col_i], xanchor='left', y=row_y_positions_5c[row_i], yanchor='middle',
            pad=dict(t=2, r=2, b=2, l=2),
            bgcolor=BUTTON_BG_ACTIVE_5C if is_default_active else BUTTON_BG_PURPLE_5C,
            bordercolor=BUTTON_BG_ACTIVE_5C if is_default_active else BUTTON_BG_PURPLE_5C,
            font=dict(color=BUTTON_TEXT_ACTIVE_5C if is_default_active else BUTTON_TEXT_PURPLE_5C, size=10)
        ))
    row_labels.append(dict(text=f"<b>{AGE_LABELS_5C[age]}</b>", xref='paper', yref='paper',
                            x=0.08, y=row_y_positions_5c[row_i], xanchor='right', yanchor='middle',
                            showarrow=False, font=dict(size=12)))

fig5c.update_layout(updatemenus=updatemenus, annotations=row_labels + legend_annotations, shapes=legend_shapes)
fig5c.show()


Enormously, far more than access itself. Among 55-74 year olds,
digital content creation ranges from 9.4% (Albania) to 92.0% (Kosovo*), an
82.7-point spread, the widest of any area/age-group combination in the
data, in a set of countries that mostly all have near-universal household
internet access by now.

## A closer look at the stricter "above basic" level

- Same map as chapter 4, now for the stricter "above basic" level defined
  in "About the data" above: "above basic" in all five DigComp areas at
  once, not just "basic or above" in each.
- Click a year/age-group combination below the map.

**Now that we know what "above basic" takes, how many actually reach it?**

In [ ]:
# Chapter 4b (closer look): same 4x3 age-group x year button grid as
# chapter 4's map, above, but for the stricter "above basic" level defined
# in "About the data" (above basic in all five DigComp areas at once, not
# just basic-or-above in each). See chapter4b_above_basic_map.py for the
# full data-provenance notes; this is its flat, shared-namespace notebook
# mirror, matching chapter4_cell.py's own pattern (suffix-free generic
# names are safe to reuse here since every later cell, chapter 5 onward,
# redefines its own copies rather than relying on leftover state).

df_raw_ch4b = pd.read_csv('digital_skills_above_basic.csv')

MISSING_COLOR = 'rgb(246,217,168)'
OCEAN_COLOR = 'rgb(223,242,232)'
UNTRACKED_COLOR = 'rgb(211,209,199)'

BUTTON_BG_PURPLE_CH4 = '#E5D8F0'
BUTTON_TEXT_PURPLE_CH4 = '#54278F'
BUTTON_BG_ACTIVE_CH4 = BUTTON_TEXT_PURPLE_CH4
BUTTON_TEXT_ACTIVE_CH4 = BUTTON_BG_PURPLE_CH4


def _pad_button_labels_ch4(labels):
    """Center-pad every label to the length of the longest one, so buttons
    in the same grid render at close to the same width with even gaps."""
    width = max(len(s) for s in labels)
    return [s.center(width) for s in labels]


AGE_GROUPS = ['16-24', '25-54', '55-74', 'All individuals (16-74)']
AGE_LABELS = {'All individuals (16-74)': 'All (16-74)', '16-24': '16-24', '25-54': '25-54', '55-74': '55-74'}
YEARS = sorted(df_raw_ch4b['year'].unique())
YEAR_LABELS_PADDED_CH4 = dict(zip(YEARS, _pad_button_labels_ch4([str(y) for y in YEARS])))

geo_ids = {f['properties']['CNTR_ID'] for f in geojson['features']}
countries_age = sorted(c for c in geo_ids if c != 'XK')
country_name = {f['properties']['CNTR_ID']: f['properties']['NAME_ENGL'] for f in geojson['features']}

combo_data = {}
for year in YEARS:
    for age in AGE_GROUPS:
        subset = df_raw_ch4b[(df_raw_ch4b['year'] == year) & (df_raw_ch4b['age_group'] == age)]
        values = subset.set_index('geo_code')['pct_above_basic_digital_skills']
        z = [values.get(c) for c in countries_age]
        hover = [
            f"{country_name.get(c, c)}<br>{v:.1f}%" if pd.notna(v) else f"{country_name.get(c, c)}<br>No data"
            for c, v in zip(countries_age, z)
        ]
        missing_z = [1 if pd.isna(v) else None for v in z]
        missing_hover = [
            f"{country_name.get(c, c)}<br>No data yet" if pd.isna(v) else ""
            for c, v in zip(countries_age, z)
        ]
        xk_v = values.get('XK')
        xk_has = pd.notna(xk_v)
        combo_data[(year, age)] = {
            'z': z, 'hover': hover, 'missing_z': missing_z, 'missing_hover': missing_hover,
            'xk_z': [xk_v if xk_has else None],
            'xk_hover': [f"{country_name.get('XK', 'XK')}<br>{xk_v:.1f}%" if xk_has else ""],
            'xk_missing_z': [None if xk_has else 1],
            'xk_missing_hover': ["" if xk_has else f"{country_name.get('XK', 'XK')}<br>No data yet"]
        }

# Defaults to the most recent year, since the answer text below only
# discusses 2025 figures.
default_year, default_age = YEARS[-1], 'All individuals (16-74)'
default = combo_data[(default_year, default_age)]
ZMAX_CH4B = 75  # actual max across the dataset is 70.6% (FI, 16-24, 2023)

fig_age = go.Figure()
fig_age.add_trace(go.Choropleth(
    locations=countries_age, z=default['z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale='Purples', zmin=0, zmax=ZMAX_CH4B, text=default['hover'], hoverinfo='text',
    marker_line_color='white', marker_line_width=0.5,
    colorbar=dict(title='% above basic<br>digital activities')
))
fig_age.add_trace(go.Choropleth(
    locations=countries_age, z=default['missing_z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]], showscale=False,
    text=default['missing_hover'], hoverinfo='text', marker_line_color='white', marker_line_width=0.3
))
fig_age.add_trace(go.Choropleth(
    locations=['XK'], z=default['xk_z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale='Purples', zmin=0, zmax=ZMAX_CH4B, showscale=False,
    text=default['xk_hover'], hoverinfo='text', marker_line_color='white', marker_line_width=0.5
))
fig_age.add_trace(go.Choropleth(
    locations=['XK'], z=default['xk_missing_z'], geojson=geojson, featureidkey='properties.CNTR_ID',
    colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]], showscale=False,
    text=default['xk_missing_hover'], hoverinfo='text', marker_line_color='white', marker_line_width=0.3
))

fig_age.update_geos(visible=False, lonaxis_range=[-25, 45], lataxis_range=[33, 72],
                    showland=True, landcolor=UNTRACKED_COLOR, showocean=True,
                    oceancolor=OCEAN_COLOR, projection_type='equirectangular',
                    domain=dict(y=[0.30, 1]))

# No chart title: see chapter4_cell.py for why.
fig_age.update_layout(
    width=1050, height=920, dragmode=False
)

LEGEND_ITEMS = [(OCEAN_COLOR, 'Ocean'), (UNTRACKED_COLOR, 'Not tracked'), (MISSING_COLOR, 'No data yet')]
legend_shapes, legend_annotations = [], []
_start_x, _y_pos, _box_w, _gap = 0.02, 0.995, 0.018, 0.13
for i, (color, label) in enumerate(LEGEND_ITEMS):
    x0 = _start_x + i * _gap
    legend_shapes.append(dict(type='rect', xref='paper', yref='paper',
                               x0=x0, x1=x0 + _box_w, y0=_y_pos - 0.035, y1=_y_pos,
                               fillcolor=color, line=dict(color='rgba(0,0,0,0.3)', width=0.5)))
    legend_annotations.append(dict(text=label, xref='paper', yref='paper',
                                    x=x0 + _box_w + 0.006, y=_y_pos - 0.017,
                                    xanchor='left', yanchor='middle', showarrow=False, font=dict(size=10)))

row_y_positions_ch4 = [0.22, 0.155, 0.09, 0.025]
col_x_positions_ch4 = [0.18, 0.36, 0.54]
combos_ch4 = [(row_i, col_i) for row_i in range(len(AGE_GROUPS)) for col_i in range(len(YEARS))]


def menu_index_ch4(row_i, col_i):
    return row_i * len(YEARS) + col_i


default_row_ch4 = AGE_GROUPS.index(default_age)
default_col_ch4 = YEARS.index(default_year)

updatemenus = []
row_labels = []
for row_i, age in enumerate(AGE_GROUPS):
    for col_i, year in enumerate(YEARS):
        d = combo_data[(year, age)]
        is_default_active = (row_i == default_row_ch4 and col_i == default_col_ch4)
        layout_update = {}
        for other_row, other_col in combos_ch4:
            idx = menu_index_ch4(other_row, other_col)
            selected = (other_row == row_i and other_col == col_i)
            layout_update[f'updatemenus[{idx}].bgcolor'] = BUTTON_BG_ACTIVE_CH4 if selected else BUTTON_BG_PURPLE_CH4
            layout_update[f'updatemenus[{idx}].bordercolor'] = BUTTON_BG_ACTIVE_CH4 if selected else BUTTON_BG_PURPLE_CH4
            layout_update[f'updatemenus[{idx}].font.color'] = BUTTON_TEXT_ACTIVE_CH4 if selected else BUTTON_TEXT_PURPLE_CH4
        updatemenus.append(dict(
            type='buttons', direction='right', showactive=False,
            buttons=[dict(
                label=YEAR_LABELS_PADDED_CH4[year], method='update',
                args=[{'z': [d['z'], d['missing_z'], d['xk_z'], d['xk_missing_z']],
                       'text': [d['hover'], d['missing_hover'], d['xk_hover'], d['xk_missing_hover']]},
                      layout_update]
            )],
            x=col_x_positions_ch4[col_i], xanchor='left', y=row_y_positions_ch4[row_i], yanchor='middle',
            pad=dict(t=2, r=2, b=2, l=2),
            bgcolor=BUTTON_BG_ACTIVE_CH4 if is_default_active else BUTTON_BG_PURPLE_CH4,
            bordercolor=BUTTON_BG_ACTIVE_CH4 if is_default_active else BUTTON_BG_PURPLE_CH4,
            font=dict(color=BUTTON_TEXT_ACTIVE_CH4 if is_default_active else BUTTON_TEXT_PURPLE_CH4, size=11)
        ))
    row_labels.append(dict(text=f"<b>{AGE_LABELS[age]}</b>", xref='paper', yref='paper',
                            x=0.08, y=row_y_positions_ch4[row_i], xanchor='right', yanchor='middle',
                            showarrow=False, font=dict(size=12)))

fig_age.update_layout(updatemenus=updatemenus, annotations=row_labels + legend_annotations, shapes=legend_shapes)

fig4b = fig_age
fig4b.show()


Far fewer reach the stricter "above basic" level, and the age gap gets
relatively worse even though it looks smaller in raw points. EU27
above-basic digital activities in 2025 ran from 43.1% for
16-24 down to 38.0% for 25-54 and just 17.1% for 55-74, only 40% of the
youngest group's rate. Under the softer basic-or-above bar, the 55-74 group
reaches 57% of the youngest group's rate instead. The point gap between
youngest and oldest is actually a bit smaller here (26.0 vs. 32.0 points),
but that is because both ends of the range shrink under the stricter
definition, not because the older group catches up.

## So, is the digital divide closing?

- Household coverage is close to a solved problem: by 2025, the
  worst-covered reporting country (Croatia, 87.9%) is closer to the best
  (Kosovo*, 99.4%) than either was to the whole continent's own 2002
  average.
- Online activities are growing across every age group, but the gap
  remains enormous: EU27 2025, 42.6% of 55–74 year olds reach the
  composite activities figure, versus 74.6% of 16–24 year olds. If
  anything, the older group is closing that gap faster in relative terms
  (2021–2025: 55–74 rose from 34.6% to 42.6%, +23% relative; 16–24 barely
  moved, 71.2% to 74.6%, +5%), likely because the youngest group is
  already near a ceiling.

Access is no longer the story; what people do with it is. Communication is
nearly universal at every age, but more advanced activities, creating
digital content, staying safe online, solving problems, didn't happen even
once in the last three months (twelve for online purchases) for most
older Europeans. Getting online was the easy part; using it for more
advanced tasks is still a work in progress.

## Export the whole story as one HTML file

Bundles all ten charts above (plus their narrative text and the closing
conclusion) into a single
self-contained HTML page. plotly.js is embedded inline, so it opens in any
browser with no software beyond that; the interactive maps do need an
internet connection, they fetch their base map layer from Plotly's CDN.
Downloads automatically when run in Colab. (The appendix below this isn't
included; it's for this notebook only.)

In [ ]:
import plotly.io as pio
import plotly.offline as pyo

INTRO_HTML = """
<section class="intro">
  <h2>About the data</h2>
  <p>Four Eurostat datasets, covering 38 European countries, 2002&ndash;2025:</p>
  <ul>
    <li><strong>Household internet access</strong> (<code>isoc_ci_in_h</code>):
    % of households with internet access, meaning a private home connection
    only, not access via work, school, hotels or cafes. 38 countries,
    2002&ndash;2025.</li>
    <li><strong>Individual internet use</strong> (<code>isoc_ci_ifp_iu</code>):
    % of individuals who used the internet in the last 3 months, anywhere,
    not just at home. 38 countries, 2002&ndash;2025; also available split
    by age group (16&ndash;24 / 25&ndash;54 / 55&ndash;74), but only from
    2021 onward.</li>
    <li><strong>Digital activities</strong> (<code>isoc_sk_dskl_i21</code>):
    % of individuals reaching "at least basic" across five activity areas, by
    age group. 37 countries, only 2021 / 2023 / 2025 (a newer, biennial
    indicator, not a continuous yearly series like the other two).</li>
    <li><strong>Digital activities, "above basic" only</strong>
    (<code>isoc_sk_dskl_i21</code>, <code>indic_is=I_DSK2_AB</code>): the
    stricter "above basic" level on its own (all five DigComp areas at once,
    not just "basic or above" in each), by age group. Same 37 countries plus
    the EU27 and euro-area aggregates, same three years as above. Used by
    the closer look near the end of chapter 4.</li>
  </ul>
  <ul>
    <li><strong>Country boundaries</strong>:
    <ul>
      <li><strong>GISCO</strong>: all countries except Kosovo*, Iceland and
      the United Kingdom.</li>
      <li><strong>OpenStreetMap</strong>:
      <ul>
        <li><strong>Kosovo*</strong>: added, because Eurostat's own boundary
        data leaves Kosovo as part of Serbia (five EU member states don't
        recognize its independence). The asterisk follows the same standard
        EU/UN convention wherever Kosovo is named elsewhere in this
        project.</li>
        <li><strong>Iceland and the United Kingdom</strong>: replaced,
        because their Eurostat coastlines didn't align cleanly with the map
        underneath, leaving visible gaps.</li>
      </ul>
      </li>
    </ul>
    </li>
  </ul>

  <h3>Why "digital activities," not "digital skills"</h3>
  <p>Eurostat's own name for the third dataset above is the <strong>Digital
  Skills Indicator 2.0</strong>, but it is not a skills test. Respondents
  aged 16&ndash;74 are simply asked yes/no whether they performed specific
  activities in the past three months (twelve months for online purchases);
  Eurostat's own methodology treats having done the activity as a proxy for
  having the underlying skill, one that only runs in one direction: 'doing
  something' implies 'being able to do it', but 'not doing it' implies
  nothing, someone may simply have no current occasion to. A retiree who
  used spreadsheets for years at work but hasn't needed to since scores the
  same as someone who never learned how. Because the actual measurement is
  activity-based, this project uses "digital activities" everywhere in its
  own writing, except when naming Eurostat's dataset or indicator directly.</p>
  <p>Activities are grouped into five DigComp 2.0 areas. What counts within
  each, and how many of an area's own activities someone needs for a
  "basic" or "above basic" score in that specific area:</p>
  <table class="areas-table">
    <tr><th>Area</th><th>Activities counted</th><th>Basic</th><th>Above basic</th></tr>
    <tr><td>Information &amp; data literacy</td><td>finding information on
    goods/services; seeking health information; reading online news;
    fact-checking online information and its sources</td>
    <td>1 activity</td><td>more than 1</td></tr>
    <tr><td>Communication &amp; collaboration</td><td>email; internet
    phone/video calls; instant messaging; social networks; expressing
    opinions on civic/political issues online; online consultations or
    voting</td><td>1 activity</td><td>more than 1</td></tr>
    <tr><td>Digital content creation</td><td>word processor; spreadsheet;
    editing photo/video/audio; copying or moving files between folders,
    devices or cloud; creating documents combining text, images, tables,
    charts; advanced spreadsheet features (formulas, macros); writing
    code</td><td>1&ndash;2 activities</td><td>3 or more</td></tr>
    <tr><td>Safety</td><td>checking a site was secure before giving personal
    data; reading privacy statements; restricting access to geographical
    location; limiting access to a social-media profile; refusing use of
    personal data for advertising; changing browser cookie settings</td>
    <td>1&ndash;2 activities</td><td>3 or more</td></tr>
    <tr><td>Problem solving</td><td>installing software/apps; changing
    device or app settings; online purchases; selling online; using online
    learning resources; internet banking; job search or applications</td>
    <td>1&ndash;2 activities</td><td>3 or more</td></tr>
  </table>
  <p>Reaching "basic" or "above basic" in one area, both measured over the
  same three-month window as the rest of this indicator, is a per-area
  score. Eurostat then combines all five per-area scores into one
  <em>overall</em> level, across all five areas at once, reusing the same
  "basic" and "above basic" labels one level up:</p>
  <table class="areas-table">
    <tr><th>Overall level</th><th>Eurostat's own definition</th></tr>
    <tr><td>No skills</td><td>"no skills" in 4 or all 5 areas, despite
    having used the internet at least once in the last 3 months.</td></tr>
    <tr><td>Low</td><td>"basic" or "above basic" level in 4 areas and "no
    skills" in 1 area (4 out of 5).</td></tr>
    <tr><td>Basic</td><td>all 5 areas at least "basic" level (some can be
    "basic" and some "above basic," but not all 5 "above basic").</td></tr>
    <tr><td>Above basic</td><td>"above basic" in all 5 areas.</td></tr>
  </table>
  <ul>
    <li>The figure used throughout most of this project is "at least basic"
    (basic and above basic combined from the table above), and it is
    conjunctive: it requires at least "basic" in <em>all
    five</em> areas at once, which means at least 5 different activities.
    Falling short of "basic" in a single area excludes someone from the
    figure entirely, however strong the other four are. The stricter "above
    basic" level is examined separately, in a closer look further down that
    compares it directly against this "at least basic" figure.</li>
    <li>As noted above, this proxy only runs one way, and that cuts deeper
    here: communication and information activities are part of daily life at
    any age, while content creation and problem solving are more tied to
    working and administrative life, so part of the age gap in this figure
    may reflect a change in circumstance rather than a genuine loss of
    skill.</li>
    <li>The composite figure is calculated across the whole age group, not
    just internet users: someone who did not use the internet at all in the
    past three months automatically fails every activity, which pulls the
    group's percentage down.</li>
    <li>The methodology changed substantially in 2021 to align with
    DigComp 2.0, so figures from 2021 onward are not comparable with
    anything earlier.</li>
  </ul>
</section>
"""

STORY_CHAPTERS = [
    dict(num=1, heading="Chapter 1: How did household internet access spread across Europe, 2002&ndash;2025?", fig=fig1, text="""
        <p>Household internet access in Europe, 2002&ndash;2025, animated by year.</p>
        <ul>
        <li>Play the animation or step through the slider.</li>
        <li>Hover a country for its name and coverage percentage that year.</li>
        <li>Amber marks a country with no data reported yet; once a country has
        reported at least once, a later gap year carries forward its last known
        shade instead of switching to amber.</li>
        </ul>
    """,
         answer="""<p>Household internet access grew significantly across the period. In 2002,
         coverage ranged from 3% (Latvia) to 58% (Netherlands). By 2025, 36 countries reported,
         ranging from 87.9% (Croatia) to 99.4% (Kosovo*); only five, Croatia, Montenegro, Bosnia
         and Herzegovina, Greece and Lithuania, were still below 90%.</p>"""),
    dict(num=2, heading="Chapter 2: Does coverage explain use?", fig=fig2a, text="""
        <p>Coverage and use could relate in three different ways, though the
        aggregate percentages alone can't fully confirm the mechanism behind
        whichever pattern shows up.</p>
        <p><strong>Hypothesis 1</strong>: use lags coverage, a household connects,
        but not every member uses it actively, or some connections even sit
        unused for a while.</p>
        <p><strong>Hypothesis 2</strong>: use leads coverage, either people go
        online outside the household, e.g. work, school, friends, internet
        cafes, before a connection ever reaches home, or a newly connected
        household has every member using it actively right away.</p>
        <p><strong>Hypothesis 3</strong>: the two just move together throughout,
        with no consistent lead or lag in either direction.</p>
        <p>Six looks at the relationship between household coverage and
        individual use, working through which of these holds.</p>
        <ul>
        <li>The year-by-year overview across the whole 2002&ndash;2025 span.</li>
        <li>A country-by-country check of how closely they actually track each other.</li>
        <li>Whether use tends to lead coverage or follow it, and whether that
        changes over time.</li>
        <li>Checking that pattern against the actual country-year data.</li>
        <li>A closer look at two individual years side by side.</li>
        <li>Where things stand today.</li>
        </ul>
    """,
         answer="""<p>It looks like individual use and household coverage move closely
         together, rising in step across the whole period (hypothesis 3). Early on,
         2002-2011, it looks more like hypothesis 2; we'll take a closer look at that
         further down. From 2012 to 2025, it looks more like hypothesis 1 instead,
         coverage running slightly ahead of use on average (mean gap -1.7 points). Let's
         check the country-by-country picture next.</p>"""),
    dict(num=None, heading="How closely do use and coverage grow together, country by country?",
         text="""
        <ul>
        <li><strong>Years reported</strong>: how many years of reported data gave
        inputs for the other two columns, out of 2002&ndash;2025.</li>
        <li><strong>Avg. gap</strong>: the average distance between individual use
        and household coverage in that country, in percentage points, regardless of
        which one was higher.</li>
        <li><strong>Correlation</strong>: how closely the two rose and fell together
        over time, from 0 (no relationship) to 1 (perfectly in step), independent of
        how far apart they actually sit.</li>
        </ul>
    """,
         body_html=ch2b_table_html_CH2B,
         answer="""<p>Use and coverage move closely together in almost every country
         (hypothesis 3). Across
         all 749 country-year observations, the median gap is 2.7 points, and 91% are within
         10 points. Every single country correlates at 0.94 or higher, most above 0.98.
         The tightest match is Switzerland
         (0.6 points, though only 6 years of data reported); the loosest is Albania (14.8 points),
         and even that still correlates at 0.963. That closeness actually grows over time: from
         2012 to 2025, the median gap narrows to 1.9 points, with 94.6% of country-years within
         10 points, both tighter than the 2002-2011 figures (median 4.9 points, 83.6% within
         10).</p>"""),
    dict(num=None, heading="A closer look at the lead-lag pattern between coverage and use", fig=fig2d,
         answer="""<p>The mean gap flips sign right around 2012: from 2002 to about 2011, the
         average country runs use ahead of coverage (mean gap +3.9 points); from 2012 on, the
         average flips to coverage ahead of use (mean gap -1.7 points). Correlation stays tight
         throughout, 0.94 or higher every year (see the country-by-country check above), so
         coverage and use never drift far apart, even as the average direction flips. That flip
         looks like a clean handoff from hypothesis 2 to hypothesis 1, but averages can hide a
         lot: the table below checks how many actual country-years back that up.</p>"""),
    dict(num=None, heading="Does either hypothesis actually hold up?",
         text="""
        <ul>
        <li><strong>Country-year</strong>: one country's data for one year, counted
        separately for each year it reported.</li>
        <li><strong>Hypothesis 2</strong>: country-years where individual use was at
        or above household coverage.</li>
        <li><strong>Hypothesis 1</strong>: country-years where household coverage
        was above individual use.</li>
        </ul>
    """,
         body_html=ch2e_table_html_CH2E,
         answer="""<p>Neither hypothesis 1 nor hypothesis 2 is confirmed in either period, only
         hypothesis 3 holds throughout. In 2002-2011, use ran ahead of coverage in a majority
         of country-years (73%, hypothesis 2), but the remaining 27% still went the other way,
         not decisive enough to call it confirmed. In 2012-2025, the lean reverses toward
         hypothesis 1 (60%), again with a large minority, 40%, still running the other way. The
         mean gap chart above is real, but it averages away roughly a quarter to two-fifths of
         countries pulling in the opposite direction in every single period.</p>"""),
    dict(num=None, heading="Fifteen years of progress in coverage and use", fig=fig2b,
         answer="""<p>In 2002, individual use was as low as 14.7% in Greece, while household
         coverage lagged even further behind: from just 3.0% in Latvia to a high of only 58.0%
         in the Netherlands, no country had reached even 60% household coverage yet. Fifteen
         years later, every reporting country had crossed the 60% mark on both fronts: the
         lowest use, 63.4% in Bulgaria, and the lowest coverage, 67.3%, also Bulgaria.</p>"""),
    dict(num=None, heading="One more year: 2025", fig=fig2c,
         answer="""<p>By 2025, every country was above 85% individual use. The lowest, Croatia,
         stood at 85.9%, just 14 points behind the highest, Ireland at 99.8%. Hypothesis 1 holds
         for most countries: in 21 of 36, coverage runs slightly ahead of use. The other 15
         still follow hypothesis 2, use at or above coverage, Montenegro, Ireland, Sweden,
         Denmark and Bosnia and Herzegovina lead that group.</p>
         <p><em>Along the way we also tried a straight correlation scatter and a
         9-panel year-by-year grid before landing on this three-chart format;
         those are kept in the notebook as a record of the paths we didn't take.</em></p>"""),
    dict(num=3, heading="Chapter 3: How much does internet use differ among generations?", fig=fig3, text="""
        <p>Household coverage alongside individual internet use by age group,
        EU27, 2021&ndash;2025.</p>
        <ul>
        <li>Household coverage in blue; individual use by age group in green,
        lightest shade is youngest.</li>
        <li>EU27 here is Eurostat's own published aggregate, calculated from
        pooled microdata across member states, which effectively weights by
        each country's population.</li>
        <li>The age breakdown only goes back to 2021, a shorter window than the
        map in chapter 1.</li>
        </ul>
    """,
         answer="""<p>The age gap in internet use is closing, driven almost entirely by the oldest
         group. Between 2021 and 2025 the gap between 16-24 and 55-74 use narrowed from 21.8 points
         to 12.6 points, as the 55-74 group rose from 75.9% to 86.2% while the two younger groups
         were already close to saturated.</p>"""),
    dict(num=4, heading="Chapter 4: How does performance of digital activities differ across generations?", fig=fig4, text="""
        <p>As a reminder: Eurostat calls this indicator "digital skills," but
        this project uses "digital activities" throughout instead, since
        respondents are only asked yes/no whether they performed specific
        activities, never tested (see "About the data" above).</p>
        <ul>
        <li>Shows the composite indicator: basic-or-above in
        <em>all five</em> DigComp areas at once (information &amp; data
        literacy, communication &amp; collaboration, digital content creation,
        safety, problem solving).</li>
        <li>Click a year/age-group combination below the map.</li>
        <li>Falling short in a single area excludes someone from this figure
        entirely, which is the main reason the 55&ndash;74 numbers look so low.</li>
        </ul>
    """,
         answer="""<p>Performance on this five-activity composite does not track plain internet use very
         closely, and the gap widens with age. The composite only counts someone once they clear all
         five activity areas, so it runs well below internet use at every age. In 2025, EU27 internet
         use exceeded it by 24.3 points for 16-24 (98.8% vs. 74.6%), 29.0
         points for 25-54 (97.5% vs. 68.6%), and 43.6 points for 55-74 (86.2% vs. 42.6%). Being
         online is close to universal at every age; clearing all five activity areas at once is not.</p>"""),
    dict(num=5, heading="Chapter 5: Which single area explains most of the generational gap?", fig=fig5a, text="""
        <ul>
        <li>Breaking the composite figure down by individual area, EU27, 2025.</li>
        <li>Areas are shown separately from the conjunctive "Overall" figure
        (right of the dotted line), which needs all five areas at once, not
        one on its own.</li>
        </ul>
    """,
         answer="""<p>Digital content creation, by a wide margin: a 36.7-point gap between 16-24 and 55-74
         (88.5% vs. 51.8%), more than double the narrowest gap, communication and collaboration,
         at 15.5 points (98.4% vs. 82.9%). Communication and collaboration is in fact the most
         common activity for every age group, and content creation is also the least common one
         for both 25-54 and 55-74 (78.3% and 51.8%). The youngest group's own weak spot is
         different: safety (83.8%), even though that's still the highest safety score of the
         three groups, their content-creation score (88.5%) is simply even higher. In fact, 9.2
         percentage points of all seniors clear digital content creation on its own but fail to
         clear at least one of the other four areas, part of why the overall composite figure
         sits at only 42.6%. As "About the data" above notes, this only measures having done the
         activity recently, not being able to; a 55-74 year old who hasn't opened a spreadsheet
         since retiring scores the same as someone who never learned how, so part of this gap
         may reflect a change in circumstance.</p>"""),
    dict(num=None, heading="Where each activity stands across countries, right now?", fig=fig5c, text="""
        <ul>
        <li>A country-level view of one skill area at a time, 2025 only (the only year
        this dataset has).</li>
        <li>Pick an age group and an activity area below the map.</li>
        <li>Purple, matching chapter 4: this is a closer look at the same measurement,
        not a different one.</li>
        </ul>
    """,
         question="Even within a single well-defined activity and a single age group, how much do countries vary?",
         answer="""<p>Enormously, far more than access itself. Among 55-74 year olds, digital content
         creation ranges from 9.4% (Albania) to 92.0% (Kosovo*), an 82.7-point spread, the
         widest of any area/age-group combination in the data, in a set of countries that
         mostly all have near-universal household internet access by now.</p>"""),
    dict(num=None, heading='A closer look at the stricter "above basic" level', fig=fig4b, text="""
        <ul>
        <li>Same map as chapter 4, now for the stricter "above basic" level
        defined in "About the data" above: "above basic" in all five DigComp
        areas at once, not just "basic or above" in each.</li>
        <li>Click a year/age-group combination below the map.</li>
        </ul>
    """,
         question="Now that we know what “above basic” takes, how many actually reach it?",
         answer="""<p>Far fewer reach the stricter "above basic" level, and the age gap gets relatively
         worse even though it looks smaller in raw points. EU27 above-basic digital activities in 2025 ran from
         43.1% for 16-24 down to 38.0% for 25-54
         and just 17.1% for 55-74, only 40% of the youngest group’s rate. Under the softer
         basic-or-above bar, the 55-74 group reaches 57% of the youngest group’s rate instead.
         The point gap between youngest and oldest is actually a bit smaller here (26.0 vs. 32.0
         points), but that is because both ends of the range shrink under the stricter definition,
         not because the older group catches up.</p>"""),
]

CONCLUSION_HTML = """
<section class="chapter">
  <div class="chapter-label">Conclusion</div>
  <h2>So, is the digital divide closing?</h2>
  <div class="narrative">
  <ul>
  <li>Household coverage is close to a solved problem: by 2025, the worst-covered
  reporting country (Croatia, 87.9%) is closer to the best (Kosovo*, 99.4%) than
  either was to the whole continent's own 2002 average.</li>
  <li>Online activities are growing across every age group, but the gap remains
  enormous: EU27 2025, 42.6% of 55&ndash;74 year olds reach the composite
  activities figure, versus 74.6% of 16&ndash;24 year olds. If anything, the
  older group is closing that gap faster in relative terms (2021&ndash;2025:
  55&ndash;74 rose from 34.6% to 42.6%, +23% relative; 16&ndash;24 barely moved,
  71.2% to 74.6%, +5%), likely because the youngest group is already near a
  ceiling.</li>
  </ul>
  <p>Access is no longer the story; what people do with it is. Communication is
  nearly universal at every age, but more advanced activities, creating digital
  content, staying safe online, solving problems, didn't happen even once in the
  last three months (twelve for online purchases) for most older Europeans.
  Getting online was the easy part; using it for more advanced tasks is still a
  work in progress.</p>
  </div>
</section>
"""

_plotly_js = pyo.get_plotlyjs()

_parts = ["""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Geospatial: How Europe grew online, 2002-2025</title>
<style>
  :root { --blue: #1F5FA6; --purple: #6A4C93; --red: #C0392B; --green: #1A7A47; --ink: #222222; --paper: #FDFDFB; }
  * { box-sizing: border-box; }
  body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
    margin: 0; padding: 0; background: var(--paper); color: var(--ink); line-height: 1.55; }
  header { padding: 56px 24px 32px; text-align: center;
    background: linear-gradient(135deg, #EAF2FB 0%, #F4EEF7 100%); border-bottom: 1px solid #E3E1D8; }
  header h1 { font-size: 1.9rem; margin: 0 0 10px; }
  header p { max-width: 640px; margin: 0 auto; color: #555; font-size: 1.02rem; }
  .chapter { max-width: 1100px; margin: 0 auto; padding: 48px 24px; border-bottom: 1px solid #E9E7DE; }
  .chapter:last-child { border-bottom: none; }
  .chapter-label { text-transform: uppercase; letter-spacing: 0.08em; font-size: 0.78rem; color: #888; margin-bottom: 6px; }
  .chapter h2 { font-size: 1.4rem; margin: 0 0 18px; }
  .chapter .narrative { max-width: 760px; margin-bottom: 22px; color: #333; }
  .chapter .narrative p { margin: 0 0 12px; }
  .chapter .narrative ul { margin: 0 0 12px; padding-left: 20px; }
  .chapter .narrative li { margin: 0 0 6px; }
  .chapter .question { max-width: 760px; margin: 0 0 18px; font-size: 1.1rem; font-weight: 600; color: #1a1a1a; }
  .figure-wrap { display: flex; justify-content: center; }
  .chapter .table-wrap { display: flex; justify-content: center; }
  .chapter table.stats-table { width: 100%; max-width: 760px; border-collapse: collapse; margin: 0; font-size: 0.88rem; }
  .chapter table.stats-table th, .chapter table.stats-table td { text-align: right; padding: 7px 10px; border-bottom: 1px solid #E3E1D8; }
  .chapter table.stats-table th:first-child, .chapter table.stats-table td:first-child { text-align: left; font-weight: 600; }
  .chapter table.stats-table th { font-weight: 600; background: #F3F1E9; }
  .chapter .answer { max-width: 760px; margin: 22px auto 0; color: #333;
    border-left: 3px solid #C9C9C9; padding-left: 16px; }
  .chapter .answer p { margin: 0 0 10px; }
  .chapter .answer p:last-child { margin-bottom: 0; }
  .intro { max-width: 760px; margin: 0 auto; padding: 40px 24px 8px; color: #333; }
  .intro h2 { font-size: 1.3rem; margin: 0 0 14px; }
  .intro h3 { font-size: 1.05rem; margin: 22px 0 10px; }
  .intro p { margin: 0 0 12px; }
  .intro ul { margin: 0 0 14px; padding-left: 20px; }
  .intro li { margin: 0 0 8px; }
  .intro li ul { margin: 8px 0 0; }
  .intro code { background: #EFEFE8; padding: 1px 5px; border-radius: 3px; font-size: 0.92em; }
  .intro table.areas-table { width: 100%; border-collapse: collapse; margin: 0 0 16px; font-size: 0.9rem; }
  .intro table.areas-table th, .intro table.areas-table td { text-align: left; padding: 8px 10px; border-bottom: 1px solid #E3E1D8; vertical-align: top; }
  .intro table.areas-table th { font-weight: 600; white-space: nowrap; }
  .intro table.areas-table td:first-child { font-weight: 600; white-space: nowrap; }
  footer { text-align: center; padding: 40px 24px 60px; color: #888; font-size: 0.85rem; }
  footer a { color: #1F5FA6; }
</style>
</head>
<body>
<script>"""]
_parts.append(_plotly_js)
_parts.append("""</script>
<header>
  <h1>How Europe grew online</h1>
  <p>Eurostat data, 2002&ndash;2025. Every chart below is interactive: hover,
  play the animations, click the buttons. No login or software needed beyond a
  browser.</p>
</header>
""")
_parts.append(INTRO_HTML)

for ch in STORY_CHAPTERS:
    # Most chapters have a plotly figure (fig=); the country-check table has
    # none, it supplies body_html directly instead (same slot in the layout).
    if ch.get('fig') is not None:
        fig_html = pio.to_html(ch['fig'], full_html=False, include_plotlyjs=False, config={'displaylogo': False})
        body_html = f'<div class="figure-wrap">{fig_html}</div>'
    else:
        body_html = ch.get('body_html', '')
    label = f"Chapter {ch['num']}" if ch['num'] else "Closer look"
    narrative_html = f'<div class="narrative">{ch["text"]}</div>' if ch.get('text') else ''
    question_html = f'<div class="question">{ch["question"]}</div>' if ch.get('question') else ''
    answer_html = f'<div class="answer">{ch["answer"]}</div>' if ch.get('answer') else ''
    _parts.append(f"""
<section class="chapter">
  <div class="chapter-label">{label}</div>
  <h2>{ch['heading']}</h2>
  {narrative_html}
  {question_html}
  {body_html}
  {answer_html}
</section>
""")

_parts.append(CONCLUSION_HTML)

_parts.append("""
<footer>
  Data: <a href="https://ec.europa.eu/eurostat">Eurostat</a> (isoc_ci_in_h, isoc_ci_ifp_iu,
  isoc_sk_dskl_i21, Eurostat's own name is "Digital Skills Indicator 2.0"; this project
  calls it "digital activities") &middot; Boundaries: GISCO / OpenStreetMap &middot;
  Built by Jana Dole&#269;kov&aacute; with Claude &middot; Geospatial project
</footer>
</body>
</html>
""")

with open('geospatial_story.html', 'w', encoding='utf-8') as f:
    f.write("".join(_parts))

print("Wrote geospatial_story.html")

from google.colab import files
files.download('geospatial_story.html')


## Appendix: paths we tried and set aside

This project keeps every real step of the investigation visible, including
the ones that didn't make the final cut, so it's clear the story above came
from systematically working through several angles rather than a single
straight line. None of the charts below are in the exported HTML story;
they're kept here as the record of what else was tried for chapter 2, and
why each one was set aside in favor of what's above.

### Chapter 2, take one: per-country scatter

In [ ]:
# Appendix, chapter 2 take one: the very first design tried for chapter 2,
# before switching to the coverage-binned min/max/mean approach used in the
# final chapters above. Reuses merged_ch2 (built in the chapter 2a cell).
#
# One dot per country per key year (2002/2008/2014/2019/2025), grayscale by
# year, plotted directly on raw coverage/use axes with a y=x reference line.
# Set aside because individual dots don't show the DISTRIBUTION across
# countries the way the binned min/max/mean chart does, and because five
# years of overlapping dots get visually busy fast.

KEY_YEARS_AP1 = [2002, 2008, 2014, 2019, 2025]
YEAR_COLORS_AP1 = {2002: '#D9D9D9', 2008: '#B3B3B3', 2014: '#8C8C8C', 2019: '#595959', 2025: '#262626'}
COVERAGE_BLUE_AP1 = '#1F5FA6'
USE_RED_AP1 = '#C0392B'

fig_ap1 = go.Figure()
fig_ap1.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines',
                              line=dict(color='rgba(0,0,0,0.35)', width=1.5, dash='dash'),
                              name='Equal coverage & use (reference)', hoverinfo='skip'))

for year in KEY_YEARS_AP1:
    sub = merged_ch2[merged_ch2['year'] == year].sort_values('geo_label')
    fig_ap1.add_trace(go.Scatter(
        x=sub['pct_households_with_internet'], y=sub['pct_individuals_used_internet_3m'],
        mode='markers', name=str(year),
        marker=dict(color=YEAR_COLORS_AP1[year], size=10, line=dict(color='white', width=0.8)),
        customdata=sub['geo_label'],
        hovertemplate='<b>%{customdata}</b><br>' + str(year) +
                       '<br>Household coverage: %{x:.1f}%<br>Individual use: %{y:.1f}%<extra></extra>'
    ))

fig_ap1.update_layout(
    title='Appendix: the original per-country scatter (superseded by the binned charts above)',
    xaxis=dict(title=dict(text='Household internet coverage (%)', font=dict(color=COVERAGE_BLUE_AP1)),
               range=[-2, 102], ticksuffix='%'),
    yaxis=dict(title=dict(text='Individual internet use (%)', font=dict(color=USE_RED_AP1)),
               range=[-2, 102], ticksuffix='%'),
    width=850, height=650, legend=dict(title='Year', orientation='v'),
    plot_bgcolor='white', margin=dict(t=80)
)
fig_ap1.update_xaxes(showgrid=True, gridcolor='#EEEEEE', zeroline=False)
fig_ap1.update_yaxes(showgrid=True, gridcolor='#EEEEEE', zeroline=False)

fig_ap1.show()


### Chapter 2, take two: the binned 2002-vs-2025 pair (title we got wrong)

In [ ]:
# Appendix, chapter 2 take two: the binned 2002-vs-2025 pair, kept exactly
# as it was first built, red/coral coloring and all, including the title we
# later found out was wrong. Reuses bin_year_ch2 / flag_emoji_ch2 /
# merged_ch2b defined in the chapter 2b cell above.
#
# The title below reads "From lagging to caught up." That is backwards: at
# the lowest coverage levels in 2002, individual use was already AHEAD of
# household coverage (about 41% average use vs. 36% average coverage that
# year), not behind it. Caught by checking the exact per-country numbers
# before publishing the claim; see chapter 2a above for the corrected,
# year-by-year version of this finding. Also the chart this session's
# flag-visibility problem was first spotted on: red/coral flags were getting
# lost against the red/coral fill, which is why the final charts use gray.

from plotly.subplots import make_subplots

YEARS_AP2 = [2002, 2025]
RANGE_FILL_AP2 = '#EFB6AD'
MEAN_LINE_AP2 = '#C0392B'

fig_ap2 = make_subplots(rows=1, cols=2, subplot_titles=[str(y) for y in YEARS_AP2], horizontal_spacing=0.08)

for i, year in enumerate(YEARS_AP2):
    col = i + 1
    df_year = merged_ch2b[merged_ch2b['year'] == year]
    bins = bin_year_ch2(df_year)
    xs = [b['mid'] for b in bins]
    mins = [b['min_val'] for b in bins]
    maxs = [b['max_val'] for b in bins]
    means = [b['mean_val'] for b in bins]

    fig_ap2.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines',
                                  line=dict(color='rgba(0,0,0,0.3)', width=1, dash='dash'),
                                  showlegend=(i == 0), name='Reference (use = coverage)'), row=1, col=col)
    fig_ap2.add_trace(go.Scatter(x=xs + xs[::-1], y=maxs + mins[::-1], fill='toself',
                                  fillcolor=RANGE_FILL_AP2, line=dict(width=0, color=RANGE_FILL_AP2),
                                  marker=dict(color=RANGE_FILL_AP2),
                                  showlegend=(i == 0), name='Range (min-max)', opacity=0.8), row=1, col=col)
    fig_ap2.add_trace(go.Scatter(x=xs, y=means, mode='lines+markers',
                                  line=dict(color=MEAN_LINE_AP2, width=2),
                                  marker=dict(size=5, color=MEAN_LINE_AP2),
                                  showlegend=(i == 0), name='Mean'), row=1, col=col)

    flag_x, flag_y, flag_text = [], [], []
    for b in bins:
        flag_x.append(b['mid']); flag_y.append(b['min_val']); flag_text.append(flag_emoji_ch2(b['min_country']))
        flag_x.append(b['mid']); flag_y.append(b['max_val']); flag_text.append(flag_emoji_ch2(b['max_country']))
    fig_ap2.add_trace(go.Scatter(x=flag_x, y=flag_y, mode='text', text=flag_text,
                                  textfont=dict(size=15), showlegend=False, hoverinfo='skip'), row=1, col=col)

    fig_ap2.update_xaxes(title='Household coverage bin (%)', range=[0, 100], row=1, col=col)
    fig_ap2.update_yaxes(title='Individual use (%)' if col == 1 else None, range=[0, 100], row=1, col=col)

fig_ap2.update_layout(
    title='Appendix: "From lagging to caught up" (title later found to be backwards, see the note above)<br>'
          '<sub>Countries grouped into ten household-coverage bands; shaded area spans min-max individual use per band, line marks the average.</sub>',
    width=1150, height=760, margin=dict(t=170), plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.18)
)

fig_ap2.show()


### Chapter 2, take three: nine snapshot years at once

In [ ]:
# Appendix, chapter 2 take three: the 9-snapshot-year grid version, approved
# in the 2026-08-14 session as chapter 2/4's chart, then dropped the next
# session in favor of a different chapter plan (and today further refined
# into the 3-chart chapter2a/b/c structure above). Reuses bin_year_ch2 /
# flag_emoji_ch2 / merged_ch2b defined in the chapter 2b cell above.
#
# Same binning logic as the charts above, computed separately for each of 9
# snapshot years, shared 0-100 axes for visual comparability. Set aside
# because a 3x3 grid asks a reader to compare nine small panels at once,
# where the final two-year and one-year charts above make the same point
# with far less to scan.

from plotly.subplots import make_subplots

YEARS_AP3 = [2002, 2005, 2008, 2011, 2014, 2017, 2020, 2023, 2025]
RANGE_FILL_AP3 = '#EFB6AD'
MEAN_LINE_AP3 = '#C0392B'

fig_ap3 = make_subplots(rows=3, cols=3, subplot_titles=[str(y) for y in YEARS_AP3],
                         horizontal_spacing=0.05, vertical_spacing=0.09)

for i, year in enumerate(YEARS_AP3):
    row, col = i // 3 + 1, i % 3 + 1
    df_year = merged_ch2b[merged_ch2b['year'] == year]
    bins = bin_year_ch2(df_year)
    if not bins:
        continue
    xs = [b['mid'] for b in bins]
    mins = [b['min_val'] for b in bins]
    maxs = [b['max_val'] for b in bins]
    means = [b['mean_val'] for b in bins]

    fig_ap3.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines',
                                  line=dict(color='rgba(0,0,0,0.3)', width=1, dash='dash'),
                                  showlegend=(i == 0), name='Reference (use = coverage)'), row=row, col=col)
    fig_ap3.add_trace(go.Scatter(x=xs + xs[::-1], y=maxs + mins[::-1], fill='toself',
                                  fillcolor=RANGE_FILL_AP3, line=dict(width=0, color=RANGE_FILL_AP3),
                                  marker=dict(color=RANGE_FILL_AP3),
                                  showlegend=(i == 0), name='Range (min-max)', opacity=0.6), row=row, col=col)
    fig_ap3.add_trace(go.Scatter(x=xs, y=means, mode='lines+markers',
                                  line=dict(color=MEAN_LINE_AP3, width=2),
                                  marker=dict(size=4, color=MEAN_LINE_AP3),
                                  showlegend=(i == 0), name='Mean'), row=row, col=col)

    flag_x, flag_y, flag_text = [], [], []
    for b in bins:
        flag_x.append(b['mid']); flag_y.append(b['min_val']); flag_text.append(flag_emoji_ch2(b['min_country']))
        flag_x.append(b['mid']); flag_y.append(b['max_val']); flag_text.append(flag_emoji_ch2(b['max_country']))
    fig_ap3.add_trace(go.Scatter(x=flag_x, y=flag_y, mode='text', text=flag_text,
                                  textfont=dict(size=13), showlegend=False, hoverinfo='skip'), row=row, col=col)

    fig_ap3.update_xaxes(range=[0, 100], row=row, col=col)
    fig_ap3.update_yaxes(range=[0, 100], row=row, col=col)

fig_ap3.update_layout(
    title='Appendix: coverage-binned distribution of individual use, nine snapshot years',
    width=1300, height=1300, plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.04)
)

fig_ap3.show()


### Before any binning: straight correlation scatters

In [ ]:
# Appendix: the very first thing tried, before any of the chapter 2 charts
# above, back in the 2026-08-14 session. Straight Pearson correlations on
# the 2025 snapshot (36 reporting countries, EU/EA aggregates excluded):
# coverage vs. use, and use vs. the composite digital-activities indicator.
# Reuses merged_ch2 (chapter 2a cell) and df_raw (chapter 4 cell).
#
# The original version of these two charts used country flags as markers,
# which turned out to be unreadable: 2025 values are nearly saturated for
# both use and the skills indicator, so almost every flag crowded into the
# same corner of the plot, badly overlapping. Rebuilt here with plain dots
# instead, which is legible, but the deeper problem stands either way: a
# single-year snapshot correlation doesn't show HOW two countries got to a
# similar point, which is exactly what the binned, multi-year charts above
# were built to show instead. That's why this angle was dropped.

use_2025_ap4 = merged_ch2[merged_ch2['year'] == 2025][['geo_code', 'geo_label',
                                                         'pct_households_with_internet',
                                                         'pct_individuals_used_internet_3m']]
skills_2025_ap4 = df_raw[(df_raw['year'] == 2025) & (df_raw['age_group'] == 'All individuals (16-74)')][
    ['geo_code', 'pct_basic_or_above_digital_skills']]

corr_hh_use_ap4 = use_2025_ap4[['pct_households_with_internet', 'pct_individuals_used_internet_3m']].corr().iloc[0, 1]
merged_use_skills_ap4 = pd.merge(use_2025_ap4, skills_2025_ap4, on='geo_code')
corr_use_skills_ap4 = merged_use_skills_ap4[['pct_individuals_used_internet_3m',
                                              'pct_basic_or_above_digital_skills']].corr().iloc[0, 1]

fig_ap4a = go.Figure()
fig_ap4a.add_trace(go.Scatter(
    x=use_2025_ap4['pct_households_with_internet'], y=use_2025_ap4['pct_individuals_used_internet_3m'],
    mode='markers', marker=dict(color='#1F5FA6', size=10, line=dict(color='white', width=0.8)),
    customdata=use_2025_ap4['geo_label'],
    hovertemplate='<b>%{customdata}</b><br>Coverage: %{x:.1f}%<br>Use: %{y:.1f}%<extra></extra>',
    showlegend=False
))
fig_ap4a.update_layout(
    title=f'Appendix: household coverage vs. individual use, 2025 (r = {corr_hh_use_ap4:.3f}, '
          f'r&sup2; = {corr_hh_use_ap4**2:.3f})<br><sub>36 countries, one dot each, EU/EA aggregates excluded</sub>',
    xaxis=dict(title='Household coverage (%)', range=[70, 102]),
    yaxis=dict(title='Individual use (%)', range=[70, 102]),
    width=700, height=550, plot_bgcolor='white'
)
fig_ap4a.show()

fig_ap4b = go.Figure()
fig_ap4b.add_trace(go.Scatter(
    x=merged_use_skills_ap4['pct_individuals_used_internet_3m'],
    y=merged_use_skills_ap4['pct_basic_or_above_digital_skills'],
    mode='markers', marker=dict(color='#54278F', size=10, line=dict(color='white', width=0.8)),
    customdata=merged_use_skills_ap4['geo_label'],
    hovertemplate='<b>%{customdata}</b><br>Use: %{x:.1f}%<br>Digital activities: %{y:.1f}%<extra></extra>',
    showlegend=False
))
fig_ap4b.update_layout(
    title=f'Appendix: individual use vs. composite digital activities, 2025 (r = {corr_use_skills_ap4:.3f}, '
          f'r&sup2; = {corr_use_skills_ap4**2:.3f})<br><sub>36 countries, one dot each, EU/EA aggregates excluded</sub>',
    xaxis=dict(title='Individual use (%)', range=[80, 102]),
    yaxis=dict(title='% with basic+ digital activities', range=[0, 100]),
    width=700, height=550, plot_bgcolor='white'
)
fig_ap4b.show()
